# FRESCO: freshness-aware coded multipath routing with conformally calibrated batch release

This notebook reproduces every number, table and figure of the manuscript on the free tier of Google Colab (CPU runtime is enough). It writes the source files, trains the neuro-fuzzy elector and the Lagrangian GAT-MAPPO release policy, tunes and runs the eight baselines, evaluates on held-out topologies and irradiance days, computes the statistics and draws the figures.

* `QUICK = True` runs a reduced pipeline (10 training episodes, 5 test episodes) to check that the pipeline works end to end; its numbers are not the paper's numbers.
* `QUICK = False` runs the full protocol of the paper; on the two vCPUs of a free runtime it takes several hours, so keep the runtime alive or run the stages in separate sessions (every stage skips work whose output file already exists).

Irradiance comes from the typical-meteorological-year files shipped inside the `pvlib` package (Greensboro NC, Miami FL, Sand Point AK), so no download is needed.

In [ ]:
!pip -q install pvlib==0.16.1 nbformat
import torch, pvlib, numpy, scipy
print(torch.__version__, pvlib.__version__)

In [ ]:
QUICK = True  # set to False for the full protocol of the paper
import os
os.environ['FRESCO_OUT'] = 'runs'; os.environ['FRESCO_FIG'] = 'figs'
os.environ['FRESCO_EPISODES'] = '10' if QUICK else '70'
if QUICK:
    os.environ['FRESCO_QUICK'] = '1'
os.makedirs('runs', exist_ok=True); os.makedirs('figs', exist_ok=True)

### `fresco_sim.py`

In [ ]:
%%writefile fresco_sim.py
"""FRESCO packet-level simulator for clustered, energy-harvesting IEEE 802.15.4 sensor networks.

One simulation step equals DT seconds. A full diurnal irradiance cycle (24 h of TMY data) is
compressed into DAY_STEPS steps, and storage capacity is scaled with the same factor, so that the
ratio between night-time drain and capacity matches an uncompressed deployment.
"""
import os
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

# ----------------------------------------------------------------------------- constants
DT = 1.0                 # decision epoch (s)
DAY_STEPS = 3600         # one compressed diurnal cycle
ROUND = 300              # cluster-head election period (steps)
V_SUP = 3.0              # supply voltage (V)
P_TX = 17.4e-3 * V_SUP   # CC2420 transmit, 0 dBm (W)
P_RX = 18.8e-3 * V_SUP   # CC2420 receive (W)
BITRATE = 250e3          # 802.15.4 O-QPSK 2.4 GHz
HDR_BYTES = 17           # MAC header + FCS + PHY header
PAYLOAD = 24             # sensing update payload (bytes)
T_W = 2.0                # relay wake-up interval for low-power listening (s)
STROBE = 0.2             # mean phase-locked strobe duration per hop burst (s)
T_W_CH = 0.25            # cluster-head wake-up interval (s)
MAX_ATT = 3              # link-layer attempts per frame
P_LISTEN_MEMBER = 0.15e-3
P_LISTEN_RELAY = 0.15e-3
P_LISTEN_CH = 3.0e-3
CAP_J = 6.0              # storage capacity (J), compressed-day units
PANEL = 1.0e-4 * 0.15    # 1 cm^2 panel, 15 % efficiency  -> W per (W/m^2)
B_MAX = 16               # cluster-head buffer limit
R_LEVELS = (0, 1, 2, 3)  # repair-packet levels
N_ACT = 1 + len(R_LEVELS)
REVIVE = 0.10            # storage fraction needed to restart after brown-out
PL0, ETA, SIG_SH = 55.0, 3.0, 4.0
NOISE_DBM, PT_DBM = -100.0, 0.0
FADE_RHO, FADE_SIG = 0.995, 1.5

DATA_DIR = None


def _tmy_dir():
    import pvlib
    return os.path.join(os.path.dirname(pvlib.__file__), 'data')


_GHI_CACHE = {}


def load_ghi(site):
    """Hourly GHI (W/m^2), 365 x 24, from the TMY files distributed with pvlib."""
    if site in _GHI_CACHE:
        return _GHI_CACHE[site]
    from pvlib.iotools import read_tmy3, read_tmy2
    d = _tmy_dir()
    if site == 'greensboro':
        df, _ = read_tmy3(os.path.join(d, '723170TYA.CSV'), map_variables=True)
        g = df['ghi'].values
    elif site == 'sandpoint':
        df, _ = read_tmy3(os.path.join(d, '703165TY.csv'), map_variables=True)
        g = df['ghi'].values
    elif site == 'miami':
        df, _ = read_tmy2(os.path.join(d, '12839.tm2'))
        g = df['GHI'].values
    else:
        raise ValueError(site)
    g = np.asarray(g, float).reshape(365, 24)
    _GHI_CACHE[site] = g
    return g


def prr_oqpsk(snr_db, frame_bytes):
    """Packet reception ratio of IEEE 802.15.4 O-QPSK (bit error expression of the standard)."""
    snr = 10.0 ** (np.asarray(snr_db, float) / 10.0)
    ber = np.zeros_like(snr)
    from math import comb
    for k in range(2, 17):
        ber = ber + ((-1) ** k) * comb(16, k) * np.exp(20.0 * snr * (1.0 / k - 1.0))
    ber = np.clip(ber * (8.0 / 15.0) * (1.0 / 16.0), 0.0, 0.5)
    return (1.0 - ber) ** (8 * frame_bytes)


def frame_time(nbytes):
    return 8.0 * (nbytes + 6) / BITRATE


class Config(dict):
    def __getattr__(self, k):
        return self[k]


def default_config(**kw):
    c = Config(N=100, density=100.0 / 100.0 ** 2, gen_period=20.0, deadline=30.0, alpha=0.05,
               site='greensboro', days=None, harvest_scale=1.0, extra_loss_db=0.0, shift_step=None,
               fail_frac=0.0, steps=DAY_STEPS, coding=True, k_paths=3)
    c.update(kw)
    return c


# ----------------------------------------------------------------------------- environment
class WSN:
    """Clustered energy-harvesting WSN with duty-cycled multi-hop multipath delivery."""

    def __init__(self, cfg, seed):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        rng = self.rng
        N = cfg.N
        self.N = N
        L = np.sqrt(N / cfg.density)
        self.L = L
        self.pos = rng.uniform(0, L, size=(N, 2))
        self.sink_pos = np.array([L / 2.0, -5.0])
        P = np.vstack([self.pos, self.sink_pos])
        self.S = N  # sink index
        d = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(-1))
        np.fill_diagonal(d, 1.0)
        self.dist = d
        sh = rng.normal(0, SIG_SH, size=(N + 1, N + 1))
        sh = (sh + sh.T) / np.sqrt(2.0)
        self.snr0 = PT_DBM - (PL0 + 10 * ETA * np.log10(np.maximum(d, 1.0)) + sh) - NOISE_DBM
        np.fill_diagonal(self.snr0, -50.0)
        fb = HDR_BYTES + PAYLOAD + B_MAX
        self.prr0 = prr_oqpsk(self.snr0, fb)
        self.adj = self.prr0 >= 0.3
        np.fill_diagonal(self.adj, False)
        # estimated link quality (beacon-based estimate with noise), refined from traffic
        self.prr_hat = np.clip(self.prr0 + rng.normal(0, 0.05, self.prr0.shape), 0.05, 1.0) * self.adj
        self.fade = np.zeros(N + 1)
        self.shade = rng.uniform(0.3, 1.0, size=N)
        # irradiance profile for this episode (compressed day)
        ghi = load_ghi(cfg.site)
        days = cfg.days if cfg.days is not None else np.arange(365)
        self.day = int(rng.choice(days))
        prof = ghi[self.day]
        t_h = np.arange(cfg.steps) * 24.0 / DAY_STEPS
        self.ghi_t = np.interp(t_h, np.arange(24) + 0.5, prof, period=24)
        self.store = rng.uniform(0.5, 0.8, size=N) * CAP_J
        self.alive = np.ones(N, bool)
        self.failed = np.zeros(N, bool)
        if cfg.fail_frac > 0:
            idx = rng.choice(N, int(round(cfg.fail_frac * N)), replace=False)
            self.fail_at = dict(zip(idx, rng.integers(cfg.steps // 4, 3 * cfg.steps // 4, size=len(idx))))
        else:
            self.fail_at = {}
        self.harv_ewma = np.zeros(N)
        self.relay_load = np.zeros(N)
        self.busy = np.zeros(N + 1)
        # traffic
        self.phase = rng.uniform(0, cfg.gen_period, size=N)
        self.next_gen = self.phase.copy()
        self.U = np.zeros(N)          # generation time of freshest delivered update
        self.t = 0
        self.is_ch = np.zeros(N, bool)
        self.chs = np.array([], int)
        self.member_of = -np.ones(N, int)
        self.buffers = {}
        self.paths = {}
        self.deliv = {}               # step -> list of (src, gen)
        self._cache = {}
        # metrics
        self.m = dict(gen=0, deliv=0, ontime=0, aoi_sum=0.0, aoi_n=0, energy=0.0, harvest=0.0,
                      bits=0.0, dead_frac=0.0, peak=[], batches=0, ctrl_frames=0, lost_mc=0,
                      released=0, lost_dead=0, lost_hop=0, shield_raise=0, infeasible=0, per_ch_ontime=[])
        self.peak_track = np.zeros(N)
        self.last_U = np.zeros(N)

    # ------------------------------------------------------------------ link/energy helpers
    def link_prr(self, i, j):
        snr = self.snr0[i, j] - self.fade[i] - self.fade[j] - self.cfg.extra_loss_db * self._shift_on()
        return prr_oqpsk(snr, HDR_BYTES + PAYLOAD + B_MAX)

    def _shift_on(self):
        s = self.cfg.shift_step
        return 1.0 if (s is not None and self.t >= s) else 0.0

    def spend(self, i, e):
        if i == self.S:
            return True
        if not self.alive[i]:
            return False
        self.store[i] -= e
        self.m['energy'] += e
        if self.store[i] <= 0:
            self.store[i] = 0.0
            self.alive[i] = False
            return False
        return True

    # ------------------------------------------------------------------ routing
    def compute_paths(self, src, k=3):
        """Up to k node-disjoint min-ETX paths from src to sink on the estimated link graph."""
        N1 = self.N + 1
        ok = np.append(self.alive & ~self.failed, True)
        w = np.where(self.adj & (self.prr_hat > 0.05), 1.0 / np.maximum(self.prr_hat, 1e-3), 0.0)
        w = w * ok[:, None] * ok[None, :]
        removed = np.zeros(N1, bool)
        out = []
        for _ in range(k):
            ww = w.copy()
            ww[removed, :] = 0
            ww[:, removed] = 0
            g = csr_matrix(ww)
            dist, pred = dijkstra(g, directed=True, indices=src, return_predecessors=True)
            if not np.isfinite(dist[self.S]):
                break
            p = [self.S]
            while p[-1] != src:
                p.append(pred[p[-1]])
            p = p[::-1]
            out.append(p)
            for v in p[1:-1]:
                removed[v] = True
            if len(p) == 2:  # direct link cannot be reused disjointly
                w[src, self.S] = 0
        return out

    def path_stats(self, p):
        """Nominal (estimated) per-packet end-to-end success and mean delay of a path."""
        key = ('p',) + tuple(p)
        if key in self._cache:
            return self._cache[key]
        q = 1.0
        etx = 0.0
        for a, b in zip(p[:-1], p[1:]):
            ph = self.prr_hat[a, b]
            q *= 1 - (1 - ph) ** MAX_ATT
            etx += 1.0 / max(ph, 1e-3)
        hops = len(p) - 1
        self._cache[key] = (q, hops, etx)
        return q, hops, etx

    # ------------------------------------------------------------------ clustering
    def node_features(self):
        """Inputs of the cluster-head elector, all scaled to [0, 1]."""
        e = self.store / CAP_J
        h = np.clip(self.harv_ewma / (PANEL * 1000.0), 0, 1)
        nb = self.adj[:self.N, :self.N]
        etx = np.where(nb, 1.0 / np.maximum(self.prr_hat[:self.N, :self.N], 1e-3), 0).sum(1) / np.maximum(nb.sum(1), 1)
        x = np.clip((etx - 1.0) / 2.0, 0, 1)
        dsink = np.clip(self.dist[:self.N, self.S] / (self.L * 1.2), 0, 1)
        q = np.clip(self.relay_load / (np.max(self.relay_load) + 1e-9), 0, 1)
        return np.stack([e, h, x, q], 1), dsink

    def elect(self, scores, K):
        """Greedy election of K heads by score with spatial suppression."""
        cand = np.where(self.alive & ~self.failed & (self.store > 0.15 * CAP_J))[0]
        if len(cand) == 0:
            cand = np.where(self.alive & ~self.failed)[0]
        rmin = 0.55 * self.L / np.sqrt(K)
        order = cand[np.argsort(-scores[cand])]
        chosen = []
        for i in order:
            if all(self.dist[i, j] >= rmin for j in chosen):
                chosen.append(i)
            if len(chosen) == K:
                break
        if len(chosen) < K:
            for i in list(order) + list(np.argsort(-self.store)):
                if i not in chosen:
                    chosen.append(int(i))
                if len(chosen) == K:
                    break
        return np.array(chosen, int)

    def set_heads(self, chs):
        old_buffers = self.buffers
        self.is_ch[:] = False
        self.is_ch[chs] = True
        self.chs = np.array(chs, int)
        self.buffers = {int(c): old_buffers.get(int(c), []) for c in chs}
        self.affiliate()
        self.paths = {int(c): self.compute_paths(int(c), self.cfg.k_paths) for c in chs}
        self._cache = {}
        # control overhead: HELLO from every node, head announcement, join
        n_al = int(self.alive.sum())
        frames = n_al + len(chs) + max(n_al - len(chs), 0)
        self.m['ctrl_frames'] += frames
        ft = frame_time(HDR_BYTES + 8)
        for i in np.where(self.alive)[0]:
            self.spend(i, P_TX * (ft + 0.01) * 2)
        self.relay_load *= 0.0

    def affiliate(self):
        alive_ch = [c for c in self.chs if self.alive[c]]
        self.member_of[:] = -1
        if not alive_ch:
            return
        ac = np.array(alive_ch)
        etx = np.where(self.adj[:self.N][:, ac], 1.0 / np.maximum(self.prr_hat[:self.N][:, ac], 1e-3), 1e6)
        etx = etx + 0.02 * self.dist[:self.N][:, ac]
        self.member_of = ac[np.argmin(etx, 1)]
        self.member_of[ac] = ac

    # ------------------------------------------------------------------ batch release
    def allocate(self, paths, B, r):
        """Inner allocation: spread B+r packets over the disjoint paths in proportion to
        w_j = q_j / h_j (per-packet success over hop count), largest-remainder rounding.
        Without repair packets the whole batch takes the best path."""
        stats = [self.path_stats(p) for p in paths]
        n = np.zeros(len(paths), int)
        tot = B + r
        if r == 0 or len(paths) == 1:
            n[0] = tot
            return n, stats
        w = np.array([s[0] / s[1] for s in stats])
        w = w / w.sum()
        raw = w * tot
        n = np.floor(raw).astype(int)
        rem = tot - n.sum()
        if rem > 0:
            n[np.argsort(-(raw - n))[:rem]] += 1
        return n, stats

    @staticmethod
    def _p_decode(n, stats, B):
        dist = np.zeros(int(n.sum()) + 1)
        dist[0] = 1.0
        for j, nj in enumerate(n):
            if nj == 0:
                continue
            q = stats[j][0]
            from math import comb
            pmf = np.array([comb(nj, k) * q ** k * (1 - q) ** (nj - k) for k in range(nj + 1)])
            dist = np.convolve(dist, pmf)[:len(dist)]
        return dist[B:].sum()

    def nominal_delay(self, n, stats):
        d = 0.0
        for nj, (q, hops, etx) in zip(n, stats):
            if nj > 0:
                d = max(d, hops * (T_W / 2.0 + STROBE) + nj * frame_time(HDR_BYTES + PAYLOAD + B_MAX) * etx)
        return max(d, 1e-3)

    def release(self, ch, r, coded=True):
        """Release the head's buffer as one batch with r repair packets. Returns batch record."""
        buf = self.buffers[ch]
        if not buf:
            return None
        paths = [p for p in self.paths.get(ch, []) if all(v == self.S or self.alive[v] for v in p[1:])]
        if not paths:
            self.paths[ch] = self.compute_paths(ch, self.cfg.k_paths)
            paths = self.paths[ch]
        B = len(buf)
        # nominal delay as seen by the deadline check of this epoch (same cached estimates)
        d_nom = self.nominal_all(ch)[0] if r in R_LEVELS else None
        self.buffers[ch] = []
        e0 = self.m['energy']
        self.m['batches'] += 1
        self.m['released'] += B
        gens = np.array([g for (_, g) in buf])
        srcs = np.array([s for (s, _) in buf])
        if not paths or not self.alive[ch]:
            return dict(B=B, T_c=np.inf, d_hat=1.0, age=self.t - gens.min(), srcs=srcs, gens=gens, r=r,
                        deliver_t=np.full(B, np.inf), energy=0.0)
        n, stats = self.allocate(paths, B, r)
        d_hat = self.nominal_delay(n, stats)
        if d_nom is not None and np.isfinite(d_nom[R_LEVELS.index(r)]):
            d_hat = float(d_nom[R_LEVELS.index(r)])
        nbytes = HDR_BYTES + PAYLOAD + (B if coded and r > 0 else 0)
        ft = frame_time(nbytes)
        # packet ids: 0..B-1 systematic originals, B..B+r-1 repairs
        ids = np.arange(B + r)
        assign = np.repeat(np.arange(len(paths)), n)
        order = np.argsort([-stats[j][0] for j in assign], kind='stable')
        assign = assign[order]  # systematic packets go to the most reliable paths first
        arrivals = []  # (time, packet ids)
        for j, p in enumerate(paths):
            pk = ids[assign == j]
            if len(pk) == 0:
                continue
            tcur = 0.0
            surv = pk.copy()
            for a, b in zip(p[:-1], p[1:]):
                if len(surv) == 0:
                    break
                ph = self.link_prr(a, b)
                att = np.zeros(len(surv), int)
                ok = np.zeros(len(surv), bool)
                for _ in range(MAX_ATT):
                    m = ~ok
                    att[m] += 1
                    ok[m] = self.rng.random(m.sum()) < ph
                self.prr_hat[a, b] = 0.97 * self.prr_hat[a, b] + 0.03 * (ok.sum() / max(att.sum(), 1))
                start = max(self.t + tcur, self.busy[a])
                wait = self.rng.uniform(0, T_W) if b != self.S else self.rng.uniform(0, 0.05)
                dur = STROBE + att.sum() * ft
                e_tx = P_TX * dur
                if not self.spend(a, e_tx):
                    self.m['lost_dead'] += len(surv)
                    surv = surv[:0]
                    break
                if b != self.S:
                    if not self.spend(b, P_RX * (ok.sum() * ft + 0.01)):
                        self.m['lost_dead'] += len(surv)
                        surv = surv[:0]
                        break
                    self.relay_load[b] += 1
                self.busy[a] = start + wait + dur
                tcur = (start - self.t) + wait + dur
                self.m['lost_hop'] += int((~ok).sum())
                surv = surv[ok]
            if len(surv):
                arrivals.append((tcur, surv))
        # decoding at the sink
        deliver_t = np.full(B, np.inf)
        arrivals.sort(key=lambda z: z[0])
        cnt = 0
        dec_t = np.inf
        got = set()
        for tt, pk in arrivals:
            for pid in pk:
                if pid < B:
                    deliver_t[pid] = min(deliver_t[pid], tt)
                got.add(int(pid))
            cnt = len(got)
            if coded and cnt >= B and dec_t == np.inf:
                dec_t = tt
        if not coded:
            # repairs are duplicates of originals, assigned cyclically
            for tt, pk in arrivals:
                for pid in pk:
                    if pid >= B:
                        o = (pid - B) % B
                        deliver_t[o] = min(deliver_t[o], tt)
        if coded:
            deliver_t = np.minimum(deliver_t, dec_t)
        T_c = deliver_t.max()
        for k in range(B):
            if np.isfinite(deliver_t[k]):
                st = self.t + int(np.ceil(deliver_t[k] / DT))
                self.deliv.setdefault(st, []).append((srcs[k], gens[k]))
                self.m['bits'] += 8 * PAYLOAD
                self.m['deliv'] += 1
                if (self.t + deliver_t[k]) - gens[k] <= self.cfg.deadline:
                    self.m['ontime'] += 1
        self._cache = {k: v for k, v in self._cache.items() if k[0] != ch and k[0] != 'p'}
        return dict(B=B, T_c=T_c, d_hat=d_hat, age=self.t - gens.min(), srcs=srcs, gens=gens, r=r, n=n,
                    deliver_t=deliver_t, energy=self.m['energy'] - e0)

    def nominal_all(self, ch):
        """Nominal completion delay d_hat(r) and decoding probability for every repair level (cached)."""
        buf = self.buffers.get(ch, [])
        B = max(len(buf), 1)
        key = (ch, B)
        if key in self._cache:
            return self._cache[key]
        paths = [p for p in self.paths.get(ch, []) if all(v == self.S or self.alive[v] for v in p[1:])]
        if not paths:
            res = (np.full(len(R_LEVELS), np.inf), np.zeros(len(R_LEVELS)))
        else:
            dh = []
            for r in R_LEVELS:
                n, stats = self.allocate(paths, B, r)
                dh.append(self.nominal_delay(n, stats))
            res = (np.array(dh), None)
        self._cache[key] = res
        return res

    def predict_bounds(self, ch, q_ratio):
        """Calibrated completion-time bound q_r * d_hat(r); q_ratio is a scalar or one value per level."""
        dh, _ = self.nominal_all(ch)
        q = np.broadcast_to(np.asarray(q_ratio, float), dh.shape)
        with np.errstate(invalid='ignore'):
            U = np.where(np.isfinite(dh), q * dh, np.inf)
        return U, dh

    # ------------------------------------------------------------------ per-step dynamics
    def step_physics(self):
        cfg = self.cfg
        t = self.t
        # failures
        for i, ft_ in list(self.fail_at.items()):
            if t >= ft_:
                self.failed[i] = True
                self.alive[i] = False
                del self.fail_at[i]
        # harvesting
        ph = PANEL * self.ghi_t[min(t, len(self.ghi_t) - 1)] * self.shade * cfg.harvest_scale
        self.harv_ewma = 0.98 * self.harv_ewma + 0.02 * ph
        self.store = np.minimum(self.store + ph * DT * ~self.failed, CAP_J)
        self.m['harvest'] += float((ph * DT).sum())
        revive = (~self.alive) & (~self.failed) & (self.store > REVIVE * CAP_J)
        if revive.any():
            self.alive[revive] = True
        # listening cost
        pl = np.where(self.is_ch, P_LISTEN_CH, P_LISTEN_MEMBER)
        e = pl * DT * self.alive
        self.store -= e
        self.m['energy'] += float(e.sum())
        died = self.alive & (self.store <= 0)
        self.store[died] = 0
        self.alive[died] = False
        if t % 30 == 0:
            self._cache = {}
        # link fading
        self.fade = FADE_RHO * self.fade + np.sqrt(1 - FADE_RHO ** 2) * self.rng.normal(0, FADE_SIG, self.N + 1)
        self.fade[self.S] = 0.0
        # re-affiliate members of dead heads
        if any(not self.alive[c] for c in self.chs):
            for c in self.chs:
                if not self.alive[c] and self.buffers.get(int(c)):
                    self.m['lost_mc'] += len(self.buffers[int(c)])
                    self.buffers[int(c)] = []
            self.affiliate()
        # generation and member -> head hop
        gen = np.where((self.next_gen <= t) & self.alive)[0]
        self.next_gen[self.next_gen <= t] += cfg.gen_period
        self.m['gen'] += len(gen)
        for i in gen:
            c = self.member_of[i]
            if c < 0:
                continue
            if c == i:
                self.buffers[int(c)].append((i, float(t)))
                continue
            p = self.link_prr(i, c)
            att = 0
            ok = False
            while att < MAX_ATT and not ok:
                att += 1
                ok = self.rng.random() < p
            ft = frame_time(HDR_BYTES + PAYLOAD)
            if not self.spend(i, P_TX * (T_W_CH * 0.05 + att * ft)):
                continue
            self.spend(c, P_RX * ft)
            if ok and self.alive[c]:
                self.buffers[int(c)].append((i, float(t)))
            else:
                self.m['lost_mc'] += 1

    def account(self):
        t = self.t
        for (s, g) in self.deliv.pop(t, []):
            if g > self.U[s]:
                self.U[s] = g
        age = t - self.U
        # peak AoI: age just before an update lowers it
        drop = self.U > self.last_U
        if drop.any():
            self.m['peak'].extend(list(self.peak_track[drop]))
        self.peak_track = age + DT
        self.last_U = self.U.copy()
        self.m['aoi_sum'] += float(age.sum())
        self.m['aoi_n'] += self.N
        self.m['dead_frac'] += float((~self.alive & ~self.failed).mean())

    def summary(self):
        m = self.m
        T = max(self.t, 1)
        gen = max(m['gen'], 1)
        pk = np.array(m['peak']) if m['peak'] else np.array([self.t])
        return dict(aoi=m['aoi_sum'] / max(m['aoi_n'], 1),
                    paoi95=float(np.percentile(pk, 95)),
                    viol=1.0 - m['ontime'] / gen,
                    pdr=m['deliv'] / gen,
                    plr=1.0 - m['deliv'] / gen,
                    e_bit=1e6 * m['energy'] / max(m['bits'], 1.0),
                    energy=m['energy'], harvest=m['harvest'],
                    brownout=m['dead_frac'] / T,
                    ctrl=m['ctrl_frames'], batches=m['batches'],
                    mean_batch=m['released'] / max(m['batches'], 1),
                    shield_raise=m['shield_raise'], infeasible=m['infeasible'],
                    lost_mc=m['lost_mc'] / gen, lost_dead=m['lost_dead'], lost_hop=m['lost_hop'], gen=m['gen'], released=m['released'])


### `fresco_agents.py`

In [ ]:
%%writefile fresco_agents.py
"""Controllers: neuro-fuzzy elector, adaptive conformal shield, GAT-MAPPO with Lagrangian duals,
and the re-implemented baselines. All controllers run inside the same simulator (fresco_sim)."""
import math
import time
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import fresco_sim as fs

torch.set_num_threads(1)
HOLD = 0  # action 0 = hold, action 1+k = release with R_LEVELS[k] repair packets
N_FEAT = 15


def k_heads(N):
    return max(3, int(math.ceil(0.1 * N)))


# ============================================================================ elector
class NeuroFuzzyElector:
    """Zero-order Takagi-Sugeno elector with Gaussian memberships (3 terms per input)."""
    TERMS = 3
    N_IN = 4

    def __init__(self, theta=None):
        self.n_rules = self.TERMS ** self.N_IN
        if theta is None:
            theta = self.default()
        self.set(theta)
        m = np.array(np.meshgrid(*[range(self.TERMS)] * self.N_IN, indexing='ij')).reshape(self.N_IN, -1).T
        self.rule_terms = m  # (R, 4)

    @classmethod
    def dim(cls):
        return 2 * cls.N_IN * cls.TERMS + cls.TERMS ** cls.N_IN

    def default(self):
        c = np.tile([0.0, 0.5, 1.0], self.N_IN)
        s = np.full(self.N_IN * self.TERMS, 0.25)
        m = np.array(np.meshgrid(*[range(self.TERMS)] * self.N_IN, indexing='ij')).reshape(self.N_IN, -1).T
        # prior: high energy, high harvest, low ETX, low relay load
        w = (m[:, 0] + m[:, 1] + (2 - m[:, 2]) + (2 - m[:, 3])) / 8.0
        return np.concatenate([c, np.log(s), w])

    def set(self, theta):
        theta = np.asarray(theta, float)
        k = self.N_IN * self.TERMS
        self.theta = theta
        self.c = theta[:k].reshape(self.N_IN, self.TERMS)
        self.s = np.exp(np.clip(theta[k:2 * k], -4, 1)).reshape(self.N_IN, self.TERMS)
        self.w = theta[2 * k:]

    def memberships(self, X):
        return np.exp(-(X[:, :, None] - self.c[None]) ** 2 / (2 * self.s[None] ** 2))  # (n,4,3)

    def score(self, X):
        mu = self.memberships(X)
        fire = np.ones((X.shape[0], self.n_rules))
        for k in range(self.N_IN):
            fire *= mu[:, k, self.rule_terms[:, k]]
        fire_n = fire / (fire.sum(1, keepdims=True) + 1e-12)
        return fire_n @ self.w

    def __call__(self, env, K):
        X, _ = env.node_features()
        return env.elect(self.score(X), K)


def heed_elect(env, K):
    X, dsink = env.node_features()
    return env.elect(X[:, 0] + 0.05 * env.rng.random(env.N), K)


class LeachElect:
    def __init__(self, p=0.1):
        self.p = p
        self.last = {}

    def __call__(self, env, K):
        rnd = env.t // fs.ROUND
        per = int(round(1 / self.p))
        elig = [i for i in range(env.N) if env.alive[i] and (rnd - self.last.get(i, -10 ** 6)) >= per]
        if len(elig) < K:
            elig = [i for i in range(env.N) if env.alive[i]]
        ch = env.rng.choice(elig, size=min(K, len(elig)), replace=False)
        for i in ch:
            self.last[int(i)] = rnd
        return np.array(ch, int)


def fbchs_elect(env, K):
    """Region-based fuzzy head selection in the style of FBCHS (five fixed regions)."""
    L = env.L
    a, b, c, d = 0.0, 0.2 * L, 0.8 * L, L
    x, y = env.pos[:, 0], env.pos[:, 1]
    reg = np.full(env.N, 4)
    reg[(x < b)] = 0
    reg[(x >= c)] = 1
    reg[(x >= b) & (x < c) & (y < b)] = 2
    reg[(x >= b) & (x < c) & (y >= c)] = 3
    e = env.store / fs.CAP_J
    dn = np.clip(env.dist[:env.N, env.S] / (1.2 * L), 0, 1)
    mu_high = np.clip((e - 0.2) / 0.6, 0, 1)
    mu_near = np.clip(1 - dn, 0, 1)
    sc = np.minimum(mu_high, mu_near) * (env.alive & (env.store > 0.15 * fs.CAP_J))
    per = int(np.ceil(K / 5))
    out = []
    for r_ in range(5):
        idx = np.where(reg == r_)[0]
        if len(idx):
            out.extend(idx[np.argsort(-sc[idx])][:per].tolist())
    return np.array(out[:K] if len(out) >= K else out, int)


# ============================================================================ conformal shield
class ConformalShield:
    """Online calibration of the per-update delivery ratio s = T_i / d_hat(r), with one score window
    per repair level r (Mondrian taxonomy) and one adaptive working level shared by all strata.
    T_i is the delivery delay of update i measured from batch release; a lost update has s = +inf.

    mode = 'aci'    adaptive conformal inference: alpha_{t+1} = alpha_t + gamma (alpha - err)
           'static' split-conformal quantile frozen after W_CAL scores per stratum
           'gauss'  mean + z * std of the finite ratios (parametric bound)
           'none'   nominal bound (ratio 1) and no enforcement
    """
    W_CAL = 300
    N_MIN = 30

    def __init__(self, alpha, mode='aci', gamma=0.002, window=1500, enforce=True, lookahead=1):
        self.alpha = alpha
        self.mode = mode
        self.gamma = gamma
        nr = len(fs.R_LEVELS)
        self.alpha_t = alpha            # one working level shared by all strata
        self.scores = [deque(maxlen=window) for _ in range(nr)]
        self.static_q = [None] * nr
        self.enforce = enforce and mode != 'none'
        self.pending = {}
        self.err_hist = []      # one entry per released update: 1 if its score exceeded the bound used
        self.miss_hist = []     # 1 if the update missed the deadline (released updates only)
        self.miss_feas_hist = []  # misses of updates released through a feasible shield decision
        self.feasible = False
        self.lookahead = lookahead

    def _q_one(self, k):
        if self.mode == 'none':
            return 1.0
        s = self.scores[k]
        n = len(s)
        if n < self.N_MIN:
            return 4.0
        arr = np.fromiter(s, float)
        if self.mode == 'gauss':
            fin = arr[np.isfinite(arr) & (arr < 1e5)]
            if len(fin) < 5:
                return 4.0
            return float(fin.mean() + _z(self.alpha) * fin.std())
        if self.mode == 'static':
            if self.static_q[k] is None:
                lev = min(1.0, math.ceil((n + 1) * (1 - self.alpha)) / n)
                qv = float(np.quantile(arr, lev, method='higher'))
                if n >= self.W_CAL:
                    self.static_q[k] = qv
                return qv if qv < 1e5 else np.inf
            return self.static_q[k] if self.static_q[k] < 1e5 else np.inf
        a = self.alpha_t
        if a <= 0:
            return np.inf
        if a >= 1:
            return 0.0
        lev = min(1.0, math.ceil((n + 1) * (1 - a)) / n)
        qv = float(np.quantile(arr, lev, method='higher'))
        return qv if qv < 1e5 else np.inf

    def q(self):
        if getattr(self, '_qc', None) is None:
            self._qc = np.array([self._q_one(k) for k in range(len(fs.R_LEVELS))])
        return self._qc

    def schedule(self, t_now, rec, q_used, deadline):
        """Feedback reaches the head when an update is delivered, or at the deadline when it is not."""
        if rec is None:
            return
        feas = self.feasible
        self.feasible = False
        k = fs.R_LEVELS.index(rec['r']) if rec['r'] in fs.R_LEVELS else len(fs.R_LEVELS) - 1
        qk = q_used[k] if np.ndim(q_used) else q_used
        for T_i, g in zip(rec['deliver_t'], rec['gens']):
            s = T_i / rec['d_hat']
            late = (t_now + T_i) - g > deadline
            t_fb = t_now + int(math.ceil(min(T_i, deadline)))
            self.pending.setdefault(t_fb, []).append((k, s, qk, late, feas))

    def tick(self, t):
        self._qc = None
        for k, s, qk, late, feas in self.pending.pop(t, []):
            err = float(s > qk)
            if self.mode == 'aci':
                self.alpha_t = self.alpha_t + self.gamma * (self.alpha - err)
            self.scores[k].append(s if np.isfinite(s) else 1e6)
            self.err_hist.append(err)
            self.miss_hist.append(float(late))
            if feas:
                self.miss_feas_hist.append(float(late))

    def apply(self, env, ch, action, deadline):
        """Executed action after the deadline check, and the calibrated ratios in force.
        Sets self.feasible for the release that follows (True if its bound meets the deadline)."""
        qr = self.q()
        self.feasible = False
        buf = env.buffers.get(ch, [])
        if not buf:
            return HOLD, qr, False
        if not self.enforce:
            return action, qr, False
        age = env.t - min(g for _, g in buf)
        U, _ = env.predict_bounds(ch, qr)
        if not np.isfinite(U).any():
            # no repair level has a finite calibrated bound: the deadline cannot be certified
            if action == HOLD:
                return HOLD, qr, False
            env.m['infeasible'] += 1
            top = len(fs.R_LEVELS)
            return top, qr, action != top
        feas = np.where(age + U <= deadline)[0]
        if action == HOLD:
            if age + self.lookahead * fs.DT + np.nanmin(U) > deadline:
                if len(feas):
                    k = feas[0]
                    self.feasible = True
                else:
                    k = int(np.argmin(U))
                    env.m['infeasible'] += 1
                env.m['shield_raise'] += 1
                return 1 + k, qr, True
            return HOLD, qr, False
        k = action - 1
        if age + U[k] <= deadline:
            self.feasible = True
            return action, qr, False
        up = feas[feas >= k] if len(feas) else feas
        if len(up):
            k2 = up[0]
            self.feasible = True
        elif len(feas):
            k2 = feas[-1]
            self.feasible = True
        else:
            k2 = int(np.argmin(U))
            env.m['infeasible'] += 1
        env.m['shield_raise'] += 1
        return 1 + k2, qr, True


def _z(a):
    from scipy.stats import norm
    return float(norm.ppf(1 - a))


# ============================================================================ observations
def ch_observation(env, ch, qr, deadline):
    buf = env.buffers.get(ch, [])
    D = deadline
    x = np.zeros(N_FEAT, np.float32)
    x[0] = len(buf) / fs.B_MAX
    if buf:
        g = np.array([g for _, g in buf])
        x[1] = min((env.t - g.min()) / D, 2.0)
        x[2] = min((env.t - g.mean()) / D, 2.0)
        U, dh = env.predict_bounds(ch, np.minimum(qr, 50.0))
        x[3] = np.clip((D - (env.t - g.min()) - U[0]) / D, -1, 1)
        x[4] = np.clip((D - (env.t - g.min()) - U[-1]) / D, -1, 1)
        x[5] = min(dh[0] / D, 2.0)
    paths = env.paths.get(ch, [])
    x[6] = len(paths) / 3.0
    if paths:
        qs = [env.path_stats(p)[0] for p in paths]
        x[7] = qs[0]
        x[8] = float(np.mean(qs))
        rel = [v for p in paths for v in p[1:-1]]
        x[11] = min(env.store[rel].min() / fs.CAP_J, 1.0) if rel else 1.0
    x[9] = env.store[ch] / fs.CAP_J
    x[10] = min(env.harv_ewma[ch] / (fs.PANEL * 1000.0), 1.0)
    ph = 2 * np.pi * (env.t % fs.DAY_STEPS) / fs.DAY_STEPS
    x[12], x[13] = np.sin(ph), np.cos(ph)
    x[14] = (env.member_of == ch).sum() / (env.N / max(len(env.chs), 1))
    return x


def ch_graph(env, k=3):
    P = env.pos[env.chs]
    K = len(env.chs)
    d = np.sqrt(((P[:, None] - P[None]) ** 2).sum(-1))
    A = np.eye(K, dtype=np.float32)
    for i in range(K):
        nn_ = np.argsort(d[i])[1:k + 1]
        A[i, nn_] = 1
    return A


# ============================================================================ networks
TAUS = (2.0, 4.0, 7.0, 11.0, 16.0, 24.0)   # candidate batch-release ages (s)
N_TAU, N_R = len(TAUS), len(fs.R_LEVELS)


class GATLayer(nn.Module):
    def __init__(self, fin, fout):
        super().__init__()
        self.W = nn.Linear(fin, fout, bias=False)
        self.a_src = nn.Parameter(torch.randn(fout) * 0.1)
        self.a_dst = nn.Parameter(torch.randn(fout) * 0.1)

    def forward(self, x, A):  # x (B,K,F), A (B,K,K)
        h = self.W(x)
        e = (h @ self.a_dst).unsqueeze(-1) + (h @ self.a_src).unsqueeze(-2)
        e = F.leaky_relu(e, 0.2).masked_fill(A < 0.5, -1e9)
        att = torch.softmax(e, -1)
        return F.elu(att @ h), att


class Actor(nn.Module):
    """Factorised release policy: a release age tau and a repair level r per batch."""

    def __init__(self, use_gat=True, hid=64):
        super().__init__()
        self.use_gat = use_gat
        self.gat = GATLayer(N_FEAT, 32) if use_gat else None
        fin = N_FEAT + (32 if use_gat else 0)
        self.body = nn.Sequential(nn.Linear(fin, hid), nn.Tanh(), nn.Linear(hid, hid), nn.Tanh())
        self.h_tau = nn.Linear(hid, N_TAU)
        self.h_r = nn.Linear(hid, N_R)
        with torch.no_grad():
            for h in (self.h_tau, self.h_r):
                h.weight.mul_(0.1)
                h.bias.zero_()

    def forward(self, x, A):
        if self.use_gat:
            z, _ = self.gat(x, A)
            x = torch.cat([x, z], -1)
        h = self.body(x)
        return self.h_tau(h), self.h_r(h)


class Critic(nn.Module):
    """Centralised critic with separate heads for return and constraint cost."""

    def __init__(self, hid=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(2 * N_FEAT, hid), nn.Tanh(), nn.Linear(hid, hid), nn.Tanh())
        self.v = nn.Linear(hid, 3)   # return, energy-budget cost, reserve cost

    def forward(self, x):
        g = x.mean(-2, keepdim=True).expand_as(x)
        h = self.body(torch.cat([x, g], -1))
        return self.v(h)


# ============================================================================ episode runner
def reserve_cost(env, ch):
    rel = {v for p in env.paths.get(ch, []) for v in p[1:-1]}
    nodes = [ch] + sorted(rel)
    st = env.store[nodes] / fs.CAP_J
    al = env.alive[nodes]
    return float(np.mean((st < 0.1) | (~al)))


def member_aoi(env, ch, D):
    mem = np.where(env.member_of == ch)[0]
    if len(mem) == 0:
        return 0.0
    return float((env.t - env.U[mem]).mean() / D)


class Controller:
    """Base controller: elector + release rule + optional shield; coded flag for RLNC."""
    name = 'base'
    coded = True
    k_paths = 3
    learner = None

    def __init__(self, elector, shield_mode='none', alpha=0.05):
        self.elector = elector
        self.shield_mode = shield_mode
        self.alpha = alpha

    def reset(self, env):
        self.shield = ConformalShield(self.alpha, self.shield_mode)

    def act(self, env, obs_ctx):
        raise NotImplementedError

    def released(self, ch):
        pass

    def flush_r(self, ch):
        return 1


def run_episode(cfg, seed, ctrl, learner=None, record=False):
    cfg = fs.Config(cfg)
    cfg['coding'] = ctrl.coded
    cfg['k_paths'] = ctrl.k_paths
    env = fs.WSN(cfg, seed)
    ctrl.learner = learner
    ctrl.reset(env)
    ctrl.log_dec = [] if record else None
    K = k_heads(env.N)
    D = cfg.deadline
    trace = dict(t=[], store_ch=[], harvest=[], alpha=[], viol_roll=[], q=[], ghi=[]) if record else None
    A = None
    if learner is not None:
        learner.begin(ctrl, K)
    for t in range(cfg.steps):
        env.t = t
        if t % fs.ROUND == 0:
            # outgoing heads hand over by flushing their buffers before the new election
            for ch in list(env.buffers):
                if env.buffers[ch] and env.alive[ch]:
                    r0 = ctrl.flush_r(ch)
                    a_f = 1 + (fs.R_LEVELS.index(r0) if r0 in fs.R_LEVELS else 0)
                    a_f, q_used, _ = ctrl.shield.apply(env, ch, a_f, D)
                    r = ctrl.repairs(env, ch, a_f) if hasattr(ctrl, 'repairs') else fs.R_LEVELS[a_f - 1]
                    rec = env.release(ch, r, coded=ctrl.coded)
                    ctrl.shield.schedule(t, rec, q_used, D)
                    ctrl.released(ch)
            chs = ctrl.elector(env, K)
            env.set_heads(chs)
            A = ch_graph(env)
            if hasattr(ctrl, 'plan'):
                ctrl.plan = {}
        env.step_physics()
        ctrl.shield.tick(t)
        qr = ctrl.shield.q()
        acts, _ = ctrl.act(env, dict(qr=qr, A=A, D=D))
        for i, ch in enumerate(env.chs):
            ch = int(ch)
            if not env.alive[ch]:
                continue
            a = acts[i]
            buf = env.buffers.get(ch, [])
            if len(buf) >= fs.B_MAX and a == HOLD:
                a = 1 + ctrl.flush_r(ch)
            a_exec, q_used, over = ctrl.shield.apply(env, ch, a, D)
            if a_exec != HOLD and env.buffers.get(ch):
                r = ctrl.repairs(env, ch, a_exec) if hasattr(ctrl, 'repairs') else fs.R_LEVELS[a_exec - 1]
                rec = env.release(ch, r, coded=ctrl.coded)
                ctrl.shield.schedule(t, rec, q_used, D)
                ctrl.released(ch)
                if learner is not None and rec is not None:
                    learner.on_release(i, rec['energy'])
            if learner is not None and over:
                learner.store_override(i)
        env.account()
        if learner is not None:
            learner.on_step(env, D)
        if record and t % 10 == 0:
            trace['t'].append(t)
            trace['store_ch'].append(float(env.store[env.chs].mean()) / fs.CAP_J)
            trace['ghi'].append(float(env.ghi_t[t]))
            trace['alpha'].append(float(ctrl.shield.alpha_t))
            mh = ctrl.shield.miss_hist[-300:]
            eh = ctrl.shield.err_hist[-300:]
            trace['viol_roll'].append(float(np.mean(mh)) if mh else np.nan)
            trace['q'].append(float(np.mean(eh)) if eh else np.nan)
    if learner is not None:
        learner.end()
    env.t = cfg.steps
    out = env.summary()
    eh = ctrl.shield.err_hist
    out['miscover'] = float(np.mean(eh)) if eh else np.nan
    out['miss_released'] = float(np.mean(ctrl.shield.miss_hist)) if ctrl.shield.miss_hist else np.nan
    mf = ctrl.shield.miss_feas_hist
    out['miss_feasible'] = float(np.mean(mf)) if mf else np.nan
    out['feasible_frac'] = len(mf) / max(len(ctrl.shield.miss_hist), 1)
    out['miss_feas_share'] = float(np.sum(mf)) / max(len(ctrl.shield.miss_hist), 1)
    if record:
        trace['decisions'] = ctrl.log_dec
        out['trace'] = trace
    return out


# ============================================================================ rule-based controllers
class Immediate(Controller):
    name = 'LEACH-Direct'
    coded = False
    k_paths = 1

    def act(self, env, ctx):
        return [1 if env.buffers.get(int(c)) else HOLD for c in env.chs], None

    def flush_r(self, ch):
        return 0


class MMSPEEDLike(Controller):
    """Immediate forwarding with probabilistic multipath duplication to reach rho."""
    name = 'HEED-MMSPEED'
    coded = False
    rho = 0.95

    def act(self, env, ctx):
        return [1 if env.buffers.get(int(c)) else HOLD for c in env.chs], None

    def repairs(self, env, ch, a):
        paths = env.paths.get(ch, [])
        if not paths:
            return 0
        qs = sorted([env.path_stats(p)[0] for p in paths], reverse=True)
        fail = 1.0
        for k, q in enumerate(qs):
            fail *= (1 - q)
            if 1 - fail >= self.rho:
                return k * len(env.buffers[ch])
        return (len(qs) - 1) * len(env.buffers[ch])

    def flush_r(self, ch):
        return 0


class FBMPBatch(Controller):
    """FBCHS heads with fixed-size or timeout batches duplicated on two disjoint paths."""
    name = 'FB-MPBatch'
    coded = False
    k_paths = 2
    batch = 8
    timeout = 20.0

    def act(self, env, ctx):
        out = []
        for c in env.chs:
            buf = env.buffers.get(int(c), [])
            rel = buf and (len(buf) >= self.batch or env.t - min(g for _, g in buf) >= self.timeout)
            out.append(1 if rel else HOLD)
        return out, None

    def repairs(self, env, ch, a):
        return len(env.buffers[ch]) if len(env.paths.get(ch, [])) > 1 else 0

    def flush_r(self, ch):
        return 0


class StaticCoded(Controller):
    name = 'SCM-RLNC'
    timeout = 10.0
    r = 2

    def act(self, env, ctx):
        out = []
        for c in env.chs:
            buf = env.buffers.get(int(c), [])
            rel = buf and env.t - min(g for _, g in buf) >= self.timeout
            out.append(1 + self.r if rel else HOLD)
        return out, None

    def flush_r(self, ch):
        return self.r


class AgeThreshold(Controller):
    """Energy-adaptive age-threshold release, single path, uncoded (MAoIG-inspired)."""
    name = 'EA-AgeThr'
    coded = False
    k_paths = 1
    tau0 = 10.0

    def act(self, env, ctx):
        out = []
        for c in env.chs:
            buf = env.buffers.get(int(c), [])
            if not buf:
                out.append(HOLD)
                continue
            e = env.store[int(c)] / fs.CAP_J
            tau = self.tau0 * (1 + (1 - e))
            out.append(1 if env.t - min(g for _, g in buf) >= tau else HOLD)
        return out, None

    def flush_r(self, ch):
        return 0


class DriftPlusPenalty(Controller):
    """Lyapunov drift-plus-penalty with energy-deficit and deadline virtual queues."""
    name = 'Lyapunov-DPP'
    V = 1.0

    def reset(self, env):
        super().reset(env)
        self.QE = {}
        self.QD = 0.0

    def act(self, env, ctx):
        D = ctx['D']
        out = []
        ph = fs.PANEL * env.ghi_t[env.t] * env.cfg.harvest_scale
        for c in env.chs:
            c = int(c)
            buf = env.buffers.get(c, [])
            qe = self.QE.get(c, 0.0)
            pc = fs.P_LISTEN_CH - ph * env.shade[c]
            self.QE[c] = max(qe + pc * fs.DT / 0.01, 0.0)
            if not buf:
                out.append(HOLD)
                continue
            g = np.array([g for _, g in buf])
            age = env.t - g.min()
            U, dh = env.predict_bounds(c, 1.0)
            best, ba = self.V * len(buf) * fs.DT / D, HOLD  # cost of holding: aging of the buffer
            hops = np.mean([len(p) - 1 for p in env.paths.get(c, [[0, 0]])])
            for k, r in enumerate(fs.R_LEVELS):
                e_cost = (hops * fs.STROBE * fs.P_TX * (1 + (r > 0))) / 0.01
                miss = float(age + dh[k] * 1.5 > D)
                cost = (self.QE[c] / 100.0) * e_cost + self.QD * miss + 0.02 * r
                if cost < best:
                    best, ba = cost, 1 + k
            out.append(ba)
            self.QD = max(self.QD + (float(age + fs.DT + dh.min() > D) - 0.05) * 0.1, 0.0)
        return out, None


# ============================================================================ learned (semi-Markov) release
class PlanController(Controller):
    """When a head's buffer opens a new batch, the agent fixes a release age tau and a repair
    level r; the batch is released once its oldest update reaches tau (or the shield intervenes)."""

    def reset(self, env):
        super().reset(env)
        self.plan = {}

    def decide(self, X, A, idx):
        raise NotImplementedError

    def act(self, env, ctx):
        chs = [int(c) for c in env.chs]
        need = [i for i, c in enumerate(chs) if env.alive[c] and env.buffers.get(c) and c not in self.plan]
        if need:
            X = np.stack([ch_observation(env, c, ctx['qr'], ctx['D']) for c in chs])
            dec = self.decide(X, ctx['A'], need)
            for i, (ti, ri, lp) in zip(need, dec):
                self.plan[chs[i]] = (ti, ri)
                if getattr(self, 'log_dec', None) is not None:
                    self.log_dec.append((int(env.t), int(ti), int(ri), float(X[i, 9]), float(X[i, 10]),
                                         float(X[i, 7])))
                if self.learner is not None:
                    self.learner.on_decision(i, X, ctx['A'], ti, ri, lp)
        acts = []
        for c in chs:
            buf = env.buffers.get(c, [])
            if not buf or c not in self.plan:
                acts.append(HOLD)
                continue
            ti, ri = self.plan[c]
            age = env.t - min(g for _, g in buf)
            acts.append(1 + ri if age >= TAUS[ti] else HOLD)
        return acts, None

    def released(self, ch):
        self.plan.pop(ch, None)

    def flush_r(self, ch):
        p = self.plan.get(ch)
        return fs.R_LEVELS[p[1]] if p else 1


class MAPPOController(PlanController):
    name = 'FRESCO'

    def __init__(self, elector, actor, shield_mode='aci', alpha=0.05, greedy=True):
        super().__init__(elector, shield_mode, alpha)
        self.actor = actor
        self.greedy = greedy

    def decide(self, X, A, idx):
        with torch.no_grad():
            lt, lr = self.actor(torch.from_numpy(X)[None], torch.from_numpy(A)[None])
        lt, lr = lt[0, idx], lr[0, idx]
        if self.greedy:
            return [(int(a), int(b), 0.0) for a, b in zip(lt.argmax(-1), lr.argmax(-1))]
        dt, dr = torch.distributions.Categorical(logits=lt), torch.distributions.Categorical(logits=lr)
        at, ar = dt.sample(), dr.sample()
        lp = (dt.log_prob(at) + dr.log_prob(ar)).numpy()
        return [(int(a), int(b), float(l)) for a, b, l in zip(at, ar, lp)]


class SMDPBuffer:
    """Per-agent-slot accumulation of discounted reward/cost between consecutive decisions."""

    def __init__(self, gamma, override_pen, pen_fixed=0.0):
        self.gamma = gamma
        self.override_pen = override_pen

    def begin(self, K):
        self.K = K
        self.open = [None] * K
        self.trans = [[] for _ in range(K)]

    def on_decision(self, i, X, A, ti, ri, lp):
        if self.open[i] is not None:
            o = self.open[i]
            o['next'] = (X, A)
            self.trans[i].append(o)
        self.open[i] = dict(X=X, A=A, ti=ti, ri=ri, lp=lp, R=0.0, C=0.0, E=0.0, k=0, next=None)

    def add(self, i, r, c):
        o = self.open[i]
        if o is None:
            return
        g = self.gamma ** o['k']
        o['R'] += g * r
        o['C'] += g * c
        o['k'] += 1

    def add_energy(self, i, e):
        o = self.open[i]
        if o is not None:
            o['E'] += (self.gamma ** o['k']) * e

    def override(self, i):
        if self.open[i] is not None:
            self.open[i]['R'] -= self.override_pen * (self.gamma ** self.open[i]['k'])

    def end(self):
        for i in range(self.K):
            if self.open[i] is not None:
                self.trans[i].append(self.open[i])
                self.open[i] = None


P_REF = 0.01   # 10 mW reference power for the energy-budget cost


class MAPPOLearner:
    """Lagrangian MAPPO on the semi-Markov batch decisions of all heads.

    Two constraints: the communication energy rate of each cluster (normalised by P_REF) must not
    exceed eps_e, and the share of the head and its relays that sit below the 10 % storage reserve
    must not exceed eps_r.  Duals follow projected ascent on the relative violation."""

    def __init__(self, actor, critic, eps_e=0.3, eps_r=0.02, lagrangian=True, fixed_pen=1.0, lr=3e-4,
                 gamma=0.99, lam=0.95, clip=0.2, epochs=6, ent=0.01, lr_dual=0.1, override_pen=0.05):
        self.actor, self.critic = actor, critic
        self.opt_a = torch.optim.Adam(actor.parameters(), lr=lr)
        self.opt_c = torch.optim.Adam(critic.parameters(), lr=lr)
        self.eps = np.array([eps_e, eps_r])
        self.lagrangian = lagrangian
        self.lmb = np.array([0.5, 0.5]) if lagrangian else np.array([fixed_pen, fixed_pen])
        self.gamma, self.lam, self.clip, self.epochs, self.ent = gamma, lam, clip, epochs, ent
        self.lr_dual = lr_dual
        self.buf = SMDPBuffer(gamma, override_pen)
        self.hist = []

    def begin(self, ctrl, K):
        self.ctrl = ctrl
        self.buf.begin(K)
        self.step_cost = []
        self.e_total = 0.0
        self.n_slot_steps = 0

    def on_decision(self, i, X, A, ti, ri, lp):
        self.buf.on_decision(i, X, A, ti, ri, lp)

    def on_release(self, i, e):
        self.buf.add_energy(i, e / (P_REF * fs.DT))
        self.e_total += e / (P_REF * fs.DT)

    def store_override(self, i):
        self.buf.override(i)

    def on_step(self, env, D):
        cs = []
        for i, c in enumerate(env.chs):
            r = -member_aoi(env, int(c), D)
            cc = reserve_cost(env, int(c))
            cs.append(cc)
            self.buf.add(i, r, cc)
        self.step_cost.append(float(np.mean(cs)))
        self.n_slot_steps += len(env.chs)

    def end(self):
        self.buf.end()

    def _gae(self, R, v, vn, gk, has_next):
        adv = np.zeros(len(R))
        pos = 0
        for seq in self.buf.trans:
            n = len(seq)
            last = 0.0
            for j in reversed(range(pos, pos + n)):
                d = R[j] + gk[j] * vn[j] - v[j]
                last = d + gk[j] * self.lam * last * has_next[j]
                adv[j] = last
            pos += n
        return adv

    def update(self):
        rows = [(i, o) for i, seq in enumerate(self.buf.trans) for o in seq]
        if not rows:
            return dict(ret=0, cost_e=0, cost_r=0, lmb_e=self.lmb[0], lmb_r=self.lmb[1])
        X = torch.from_numpy(np.stack([o['X'] for _, o in rows]))
        A = torch.from_numpy(np.stack([o['A'] for _, o in rows]))
        idx = torch.tensor([i for i, _ in rows])
        has_next = np.array([o['next'] is not None for _, o in rows], float)
        Xn = torch.from_numpy(np.stack([o['next'][0] if o['next'] is not None else o['X'] for _, o in rows]))
        ar = torch.arange(len(rows))
        with torch.no_grad():
            v = self.critic(X)[ar, idx].numpy()      # (n,3)
            vn = self.critic(Xn)[ar, idx].numpy() * has_next[:, None]
        R = np.array([o['R'] for _, o in rows])
        E = np.array([o['E'] for _, o in rows])
        C = np.array([o['C'] for _, o in rows])
        gk = self.gamma ** np.array([o['k'] for _, o in rows])
        adv = [self._gae(Z, v[:, k], vn[:, k], gk, has_next) for k, Z in enumerate((R, E, C))]
        rets = [adv[k] + v[:, k] for k in range(3)]
        j_e = self.e_total / max(self.n_slot_steps, 1)
        j_r = float(np.mean(self.step_cost))
        if self.lagrangian:
            self.lmb = np.maximum(0.0, self.lmb + self.lr_dual * np.clip(np.array([j_e, j_r]) / self.eps - 1.0, -1.0, 1.0))
        a = adv[0] - self.lmb[0] * adv[1] - self.lmb[1] * adv[2]
        a = (a - a.mean()) / (a.std() + 1e-8)
        adv_t = torch.from_numpy(a).float()
        ret_t = torch.from_numpy(np.stack(rets, 1)).float()
        ti = torch.tensor([o['ti'] for _, o in rows])
        ri = torch.tensor([o['ri'] for _, o in rows])
        lp_old = torch.tensor([o['lp'] for _, o in rows]).float()
        T = len(rows)
        mb = 512
        for _ in range(self.epochs):
            perm = torch.randperm(T)
            for s in range(0, T, mb):
                b = perm[s:s + mb]
                bb = torch.arange(len(b))
                lt, lr = self.actor(X[b], A[b])
                dt = torch.distributions.Categorical(logits=lt[bb, idx[b]])
                dr = torch.distributions.Categorical(logits=lr[bb, idx[b]])
                lp = dt.log_prob(ti[b]) + dr.log_prob(ri[b])
                ratio = torch.exp(lp - lp_old[b])
                a_ = adv_t[b]
                la = -torch.min(ratio * a_, torch.clamp(ratio, 1 - self.clip, 1 + self.clip) * a_).mean()
                la = la - self.ent * (dt.entropy() + dr.entropy()).mean()
                self.opt_a.zero_grad()
                la.backward()
                nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
                self.opt_a.step()
                pv = self.critic(X[b])[bb, idx[b]]
                lc = F.mse_loss(pv, ret_t[b])
                self.opt_c.zero_grad()
                lc.backward()
                nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
                self.opt_c.step()
        info = dict(ret=float(R.mean()), cost_e=j_e, cost_r=j_r, lmb_e=float(self.lmb[0]),
                    lmb_r=float(self.lmb[1]), n=T)
        self.hist.append(info)
        return info


def train_mappo(cfg, elector, seeds, use_gat=True, lagrangian=True, coded=True, shield_mode='aci',
                episodes=60, eps_e=0.3, eps_r=0.02, fixed_pen=1.0, torch_seed=0, log=None):
    torch.manual_seed(torch_seed)
    np.random.seed(torch_seed)
    actor, critic = Actor(use_gat), Critic()
    learner = MAPPOLearner(actor, critic, eps_e=eps_e, eps_r=eps_r, lagrangian=lagrangian, fixed_pen=fixed_pen)
    ctrl = MAPPOController(elector, actor, shield_mode=shield_mode, alpha=cfg['alpha'], greedy=False)
    ctrl.coded = coded
    for ep in range(episodes):
        t0 = time.time()
        res = run_episode(cfg, int(seeds[ep % len(seeds)]), ctrl, learner=learner)
        info = learner.update()
        info.update(ep=ep, aoi=res['aoi'], viol=res['viol'], brown=res['brownout'], ebit=res['e_bit'],
                    sec=time.time() - t0)
        if log is not None:
            log.append(info)
    return actor, critic, learner


# ============================================================================ independent DQN (SMDP)
class QNet(nn.Module):
    def __init__(self, hid=64):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(N_FEAT, hid), nn.ReLU(), nn.Linear(hid, hid), nn.ReLU(),
                               nn.Linear(hid, N_TAU * N_R))

    def forward(self, x):
        return self.f(x)


class DQNController(PlanController):
    name = 'IQL-DQN'

    def __init__(self, elector, qnet, eps=0.0):
        super().__init__(elector, 'none')
        self.q = qnet
        self.eps = eps
        self.rng = np.random.default_rng(0)

    def decide(self, X, A, idx):
        with torch.no_grad():
            a = self.q(torch.from_numpy(X[idx])).argmax(-1).numpy()
        if self.eps > 0:
            m = self.rng.random(len(a)) < self.eps
            a[m] = self.rng.integers(0, N_TAU * N_R, m.sum())
        return [(int(x // N_R), int(x % N_R), 0.0) for x in a]


class DQNLearner:
    def __init__(self, qnet, pen=1.0, gamma=0.99, lr=5e-4):
        self.q = qnet
        self.tq = QNet()
        self.tq.load_state_dict(qnet.state_dict())
        self.opt = torch.optim.Adam(qnet.parameters(), lr=lr)
        self.rb = deque(maxlen=100000)
        self.pen, self.gamma = pen, gamma
        self.buf = SMDPBuffer(gamma, 0.0)
        self.n_upd = 0

    def begin(self, ctrl, K):
        self.buf.begin(K)

    def on_decision(self, i, X, A, ti, ri, lp):
        o = self.buf.open[i]
        self.buf.on_decision(i, X, A, ti, ri, lp)
        if o is not None:
            self.rb.append((o['X'][i], o['ti'] * N_R + o['ri'], o['R'] - self.pen * (o['C'] + o['E']), o['k'], X[i], 1.0))
            self._train()

    def store_override(self, i):
        pass

    def on_release(self, i, e):
        self.buf.add_energy(i, e / (P_REF * fs.DT))

    def on_step(self, env, D):
        for i, c in enumerate(env.chs):
            self.buf.add(i, -member_aoi(env, int(c), D), reserve_cost(env, int(c)))

    def end(self):
        for i in range(self.buf.K):
            o = self.buf.open[i]
            if o is not None:
                self.rb.append((o['X'][i], o['ti'] * N_R + o['ri'], o['R'] - self.pen * (o['C'] + o['E']), o['k'], o['X'][i], 0.0))
        self.buf.end()

    def _train(self):
        if len(self.rb) < 1000:
            return
        b = [self.rb[k] for k in np.random.randint(0, len(self.rb), 128)]
        x = torch.tensor(np.stack([z[0] for z in b]))
        a = torch.tensor([z[1] for z in b]).long()
        r_ = torch.tensor([z[2] for z in b]).float()
        k = torch.tensor([z[3] for z in b]).float()
        x2 = torch.tensor(np.stack([z[4] for z in b]))
        nd = torch.tensor([z[5] for z in b]).float()
        with torch.no_grad():
            a2 = self.q(x2).argmax(-1, keepdim=True)
            y = r_ + nd * (self.gamma ** k) * self.tq(x2).gather(1, a2).squeeze(1)
        qv = self.q(x).gather(1, a[:, None]).squeeze(1)
        loss = F.smooth_l1_loss(qv, y)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        self.n_upd += 1
        if self.n_upd % 500 == 0:
            self.tq.load_state_dict(self.q.state_dict())


def train_dqn(cfg, elector, seeds, episodes=40, pen=1.0, torch_seed=0, log=None):
    torch.manual_seed(torch_seed)
    np.random.seed(torch_seed)
    q = QNet()
    learner = DQNLearner(q, pen=pen)
    for ep in range(episodes):
        eps = max(0.05, 1.0 - ep / (0.6 * episodes))
        ctrl = DQNController(elector, q, eps=eps)
        res = run_episode(cfg, int(seeds[ep % len(seeds)]), ctrl, learner=learner)
        if log is not None:
            log.append(dict(ep=ep, aoi=res['aoi'], viol=res['viol'], brown=res['brownout']))
    return q


# ============================================================================ CEM for the elector
class FixedReleaseShielded(Controller):
    """Release rule used while the elector is tuned: timeout plus one repair packet, ACI shield."""
    name = 'fixed'
    timeout = 12.0

    def act(self, env, ctx):
        out = []
        for c in env.chs:
            buf = env.buffers.get(int(c), [])
            rel = buf and env.t - min(g for _, g in buf) >= self.timeout
            out.append(2 if rel else HOLD)
        return out, None

    def flush_r(self, ch):
        return 1


def elector_fitness(res, D):
    return -(res['aoi'] / D + 3.0 * res['viol'] + 10.0 * res['brownout'])


def train_elector_cem(cfg, seeds, iters=8, pop=16, elite=4, rng_seed=0, log=None, pool=None):
    rng = np.random.default_rng(rng_seed)
    base = NeuroFuzzyElector()
    mu = base.default().copy()
    sd = np.concatenate([np.full(12, 0.15), np.full(12, 0.3), np.full(81, 0.3)])
    best = (-np.inf, mu.copy())
    for it in range(iters):
        cand = mu + sd * rng.standard_normal((pop, len(mu)))
        cand[0] = mu
        sds = [int(seeds[(it * 2 + k) % len(seeds)]) for k in range(2)]
        jobs = [(cfg, c, sds) for c in cand]
        fit = np.array(pool.map(_cem_eval, jobs) if pool else [_cem_eval(j) for j in jobs])
        order = np.argsort(-fit)
        el = cand[order[:elite]]
        mu = 0.7 * el.mean(0) + 0.3 * mu
        sd = 0.7 * el.std(0) + 0.3 * sd * 0.8 + 0.01
        if fit[order[0]] > best[0]:
            best = (fit[order[0]], cand[order[0]].copy())
        if log is not None:
            log.append(dict(it=it, best=float(fit[order[0]]), mean=float(fit.mean())))
    return NeuroFuzzyElector(mu), NeuroFuzzyElector(best[1])


def _cem_eval(job):
    cfg, theta, sds = job
    el = NeuroFuzzyElector(theta)
    f = 0.0
    for s in sds:
        res = run_episode(cfg, s, FixedReleaseShielded(el, 'aci', cfg['alpha']))
        f += elector_fitness(res, cfg['deadline'])
    return f / len(sds)


### `train_all.py`

In [ ]:
%%writefile train_all.py
"""Stage-wise training: (1) CEM for the neuro-fuzzy elector, (2) Lagrangian GAT-MAPPO and the
learning-based ablations/baselines, (3) grid tuning of rule-based baselines on training seeds."""
import json, os, sys, time, pickle
import multiprocessing as mp
import numpy as np
import torch
import fresco_sim as fs
import fresco_agents as fa

OUT = os.environ.get('FRESCO_OUT', '.')
os.makedirs(OUT, exist_ok=True)
TRAIN_DAYS = list(range(0, 365, 2))
TRAIN_SEEDS = list(range(100, 200))
TUNE_SEEDS = [200, 201, 202, 203]
EPISODES = int(os.environ.get('FRESCO_EPISODES', 70))


def base_cfg(**kw):
    return fs.default_config(days=TRAIN_DAYS, **kw)


def stage_cem():
    path = f'{OUT}/elector.npy'
    if os.path.exists(path):
        return
    log = []
    t0 = time.time()
    with mp.Pool(2) as pool:
        mean_el, best_el = fa.train_elector_cem(base_cfg(), TRAIN_SEEDS, iters=8, pop=16, elite=4, log=log, pool=pool)
    np.save(path, mean_el.theta)
    json.dump(dict(log=log, sec=time.time() - t0), open(f'{OUT}/elector_log.json', 'w'))
    print('CEM done', time.time() - t0, log[-1], flush=True)


def load_elector():
    return fa.NeuroFuzzyElector(np.load(f'{OUT}/elector.npy'))


def _eps_e():
    p = f'{OUT}/selected_eps.json'
    return json.load(open(p))['eps_e'] if os.path.exists(p) else float(os.environ.get('FRESCO_EPS_E', 0.3))
VARIANTS = {
    # name: (elector, use_gat, lagrangian, coded, shield)
    'fresco_e45':    ('nf', True, True, True, 'aci'),
    'fresco_e60':    ('nf', True, True, True, 'aci'),
    'fresco':        ('nf', True, True, True, 'aci'),
    'fresco_s2':     ('nf', True, True, True, 'aci'),
    'abl_nogat':     ('nf', False, True, True, 'aci'),
    'abl_nolag':     ('nf', True, False, True, 'aci'),
    'abl_nocode':    ('nf', True, True, False, 'aci'),
    'abl_heed':      ('heed', True, True, True, 'aci'),
    'mappo_pen':     ('heed', False, False, True, 'none'),
}


def train_variant(name):
    torch.set_num_threads(1)
    path = f'{OUT}/actor_{name}.pt'
    if os.path.exists(path):
        return name
    el_kind, gat, lag, coded, shield = VARIANTS[name]
    el = load_elector() if el_kind == 'nf' else fa.heed_elect
    log = []
    t0 = time.time()
    ts = 1 if name.endswith('_s2') else 0
    eps_e = {'fresco_e20': 0.2, 'fresco_e45': 0.45, 'fresco_e60': 0.6}.get(name, _eps_e())
    actor, critic, learner = fa.train_mappo(base_cfg(), el, TRAIN_SEEDS, use_gat=gat, lagrangian=lag, coded=coded,
                                            shield_mode=shield, episodes=EPISODES, torch_seed=ts, eps_e=eps_e,
                                            log=log)
    torch.save(actor.state_dict(), path)
    json.dump(dict(log=log, sec=time.time() - t0), open(f'{OUT}/trainlog_{name}.json', 'w'))
    print('trained', name, round(time.time() - t0), log[-1], flush=True)
    return name


def train_dqn_job(_):
    torch.set_num_threads(1)
    path = f'{OUT}/qnet_dqn.pt'
    if os.path.exists(path):
        return 'dqn'
    log = []
    t0 = time.time()
    q = fa.train_dqn(base_cfg(), fa.heed_elect, TRAIN_SEEDS, episodes=40, log=log)
    torch.save(q.state_dict(), path)
    json.dump(dict(log=log, sec=time.time() - t0), open(f'{OUT}/trainlog_dqn.json', 'w'))
    print('trained dqn', round(time.time() - t0), log[-1], flush=True)
    return 'dqn'


GRIDS = {
    'MMSPEEDLike': ('rho', [0.9, 0.95, 0.99]),
    'FBMPBatch': ('timeout', [5.0, 10.0, 20.0, 30.0]),
    'StaticCoded': ('timeout', [3.0, 5.0, 10.0, 15.0, 20.0]),
    'AgeThreshold': ('tau0', [2.0, 3.0, 5.0, 10.0, 15.0]),
    'DriftPlusPenalty': ('V', [0.3, 1.0, 3.0, 10.0, 30.0, 100.0]),
}
ELECT = {'MMSPEEDLike': 'heed', 'FBMPBatch': 'fbchs', 'StaticCoded': 'heed', 'AgeThreshold': 'heed', 'DriftPlusPenalty': 'heed'}


def _tune_eval(job):
    cname, attr, val = job
    C = getattr(fa, cname)
    el = {'heed': fa.heed_elect, 'fbchs': fa.fbchs_elect}[ELECT[cname]]
    fit = []
    for s in TUNE_SEEDS:
        ctrl = C(el, 'none')
        setattr(ctrl, attr, val)
        r = fa.run_episode(base_cfg(), s, ctrl)
        fit.append(fa.elector_fitness(r, 30.0))
    return cname, val, float(np.mean(fit))


def stage_tune():
    path = f'{OUT}/tuned.json'
    if os.path.exists(path):
        return
    jobs = [(c, a, v) for c, (a, vals) in GRIDS.items() for v in vals]
    with mp.Pool(2) as pool:
        res = pool.map(_tune_eval, jobs)
    best = {}
    for c, v, f in res:
        if c not in best or f > best[c][1]:
            best[c] = (v, f)
    json.dump(dict(best={c: dict(attr=GRIDS[c][0], value=v, fit=f) for c, (v, f) in best.items()},
                   all=[list(r) for r in res]), open(path, 'w'), indent=1)
    print('tuned', best, flush=True)


if __name__ == '__main__':
    stage = sys.argv[1] if len(sys.argv) > 1 else 'all'
    if stage in ('cem', 'all'):
        stage_cem()
    if stage in ('tune', 'all'):
        stage_tune()
    if stage == 'eps':
        with mp.Pool(2) as pool:
            pool.map(train_variant, ['fresco', 'fresco_e45', 'fresco_e60'])
    if stage in ('rl', 'all'):
        names = [n for n in VARIANTS if not n.startswith('fresco_e')]
        with mp.Pool(2) as pool:
            jobs = [pool.apply_async(train_variant, (n,)) for n in names] + [pool.apply_async(train_dqn_job, (0,))]
            for j in jobs:
                j.get()


### `select_eps.py`

In [ ]:
%%writefile select_eps.py
"""Select the energy budget eps_E on the tuning seeds with the same fitness used for baseline tuning,
then promote the chosen policy to actor_fresco.pt."""
import json, os, shutil
import numpy as np
import torch
import fresco_sim as fs
import fresco_agents as fa

torch.set_num_threads(1)
OUT = os.environ.get('FRESCO_OUT', '.')
CANDS = {'fresco_e30': 0.30, 'fresco_e45': 0.45, 'fresco_e60': 0.60}
TUNE_SEEDS = [200, 201, 202, 203]


def main():
    if os.path.exists(f'{OUT}/selected_eps.json'):
        print(open(f'{OUT}/selected_eps.json').read())
        return
    if os.path.exists(f'{OUT}/actor_fresco.pt') and not os.path.exists(f'{OUT}/actor_fresco_e30.pt'):
        shutil.move(f'{OUT}/actor_fresco.pt', f'{OUT}/actor_fresco_e30.pt')
        shutil.move(f'{OUT}/trainlog_fresco.json', f'{OUT}/trainlog_fresco_e30.json')
    el = fa.NeuroFuzzyElector(np.load(f'{OUT}/elector.npy'))
    out = {}
    for name, eps in CANDS.items():
        a = fa.Actor(True)
        a.load_state_dict(torch.load(f'{OUT}/actor_{name}.pt'))
        a.eval()
        rows = [fa.run_episode(fs.default_config(days=list(range(0, 365, 2))), s, fa.MAPPOController(el, a, 'aci', 0.05))
                for s in TUNE_SEEDS]
        out[name] = dict(eps_e=eps, F=float(np.mean([fa.elector_fitness(r, 30.0) for r in rows])),
                         aoi=float(np.mean([r['aoi'] for r in rows])), viol=float(np.mean([r['viol'] for r in rows])),
                         e_bit=float(np.mean([r['e_bit'] for r in rows])),
                         brownout=float(np.mean([r['brownout'] for r in rows])))
        print(name, out[name], flush=True)
    best = max(out, key=lambda k: out[k]['F'])
    shutil.copy(f'{OUT}/actor_{best}.pt', f'{OUT}/actor_fresco.pt')
    shutil.copy(f'{OUT}/trainlog_{best}.json', f'{OUT}/trainlog_fresco.json')
    json.dump(dict(selected=best, eps_e=CANDS[best], candidates=out), open(f'{OUT}/selected_eps.json', 'w'), indent=1)
    print('selected', best)


if __name__ == '__main__':
    main()


### `run_eval.py`

In [ ]:
%%writefile run_eval.py
"""Evaluation of FRESCO, the eight baselines and the ablations on held-out seeds and irradiance days."""
import json, os, sys, time
import multiprocessing as mp
import numpy as np
import torch
import fresco_sim as fs
import fresco_agents as fa

OUT = os.environ.get('FRESCO_OUT', '.')
TEST_DAYS = list(range(1, 365, 2))
TEST_SEEDS = list(range(1000, 1030))
if os.environ.get('FRESCO_QUICK'):
    TEST_SEEDS = list(range(1000, 1005))
TUNED = None


def tuned():
    global TUNED
    if TUNED is None:
        TUNED = json.load(open(f'{OUT}/tuned.json'))['best']
    return TUNED


def _actor(name, gat=True):
    a = fa.Actor(gat)
    a.load_state_dict(torch.load(f'{OUT}/actor_{name}.pt'))
    a.eval()
    return a


def _el():
    return fa.NeuroFuzzyElector(np.load(f'{OUT}/elector.npy'))


def _rule(cname, el):
    C = getattr(fa, cname)
    c = C(el, 'none')
    t = tuned().get(cname)
    if t:
        setattr(c, t['attr'], t['value'])
    return c


def make(name, alpha=0.05):
    torch.set_num_threads(1)
    if name == 'FRESCO':
        return fa.MAPPOController(_el(), _actor('fresco'), 'aci', alpha)
    if name == 'LEACH-Direct':
        c = fa.Immediate(fa.LeachElect(), 'none'); c.name = name; return c
    if name == 'HEED-MMSPEED':
        return _rule('MMSPEEDLike', fa.heed_elect)
    if name == 'FB-MPBatch':
        return _rule('FBMPBatch', fa.fbchs_elect)
    if name == 'SCM-RLNC':
        return _rule('StaticCoded', fa.heed_elect)
    if name == 'EA-AgeThr':
        return _rule('AgeThreshold', fa.heed_elect)
    if name == 'Lyapunov-DPP':
        return _rule('DriftPlusPenalty', fa.heed_elect)
    if name == 'IQL-DQN':
        q = fa.QNet(); q.load_state_dict(torch.load(f'{OUT}/qnet_dqn.pt')); q.eval()
        return fa.DQNController(fa.heed_elect, q)
    if name == 'MAPPO-Penalty':
        return fa.MAPPOController(fa.heed_elect, _actor('mappo_pen', False), 'none', alpha)
    # ablations
    if name == 'w/o NF elector':
        return fa.MAPPOController(fa.heed_elect, _actor('abl_heed'), 'aci', alpha)
    if name == 'w/o RLNC':
        c = fa.MAPPOController(_el(), _actor('abl_nocode'), 'aci', alpha); c.coded = False; return c
    if name == 'w/o learned release':
        return fa.FixedReleaseShielded(_el(), 'aci', alpha)
    if name == 'w/o shield':
        return fa.MAPPOController(_el(), _actor('fresco'), 'none', alpha)
    if name == 'Gaussian bound':
        return fa.MAPPOController(_el(), _actor('fresco'), 'gauss', alpha)
    if name == 'Static split CP':
        return fa.MAPPOController(_el(), _actor('fresco'), 'static', alpha)
    if name == 'w/o GAT':
        return fa.MAPPOController(_el(), _actor('abl_nogat', False), 'aci', alpha)
    if name == 'w/o Lagrangian':
        return fa.MAPPOController(_el(), _actor('abl_nolag'), 'aci', alpha)
    if name == 'FRESCO (seed 2)':
        return fa.MAPPOController(_el(), _actor('fresco_s2'), 'aci', alpha)
    raise KeyError(name)


MAIN = ['FRESCO', 'LEACH-Direct', 'HEED-MMSPEED', 'FB-MPBatch', 'SCM-RLNC', 'EA-AgeThr', 'Lyapunov-DPP',
        'IQL-DQN', 'MAPPO-Penalty']
ABL = ['w/o NF elector', 'w/o RLNC', 'w/o learned release', 'w/o shield', 'Gaussian bound',
       'Static split CP', 'w/o GAT', 'w/o Lagrangian', 'FRESCO (seed 2)']


def _job(job):
    name, cfgkw, seed, alpha, record = job
    cfg = fs.default_config(**cfgkw)
    t0 = time.time()
    ctrl = make(name, alpha)
    r = fa.run_episode(cfg, seed, ctrl, record=record)
    r['sec'] = time.time() - t0
    r['method'] = name
    r['seed'] = seed
    r.update({f'cfg_{k}': v for k, v in cfgkw.items() if k != 'days'})
    return {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in r.items()}


def run(jobs, tag):
    path = f'{OUT}/eval_{tag}.json'
    if os.path.exists(path):
        return json.load(open(path))
    t0 = time.time()
    with mp.Pool(2) as pool:
        res = pool.map(_job, jobs, chunksize=1)
    json.dump(res, open(path, 'w'))
    print(tag, len(res), 'episodes', round(time.time() - t0), 's', flush=True)
    return res


def J(methods, seeds, alpha=0.05, record=False, **cfgkw):
    cfgkw.setdefault('days', TEST_DAYS)
    return [(m, cfgkw, s, alpha, record) for m in methods for s in seeds]


if __name__ == '__main__':
    which = sys.argv[1:] or ['main', 'abl', 'alpha', 'deadline', 'load', 'harvest', 'scale', 'site', 'shift', 'fail', 'trace']
    S10 = TEST_SEEDS[:10]
    S15 = TEST_SEEDS[:15]
    KEY = ['FRESCO', 'FB-MPBatch', 'SCM-RLNC', 'EA-AgeThr', 'Lyapunov-DPP']
    if 'main' in which:
        run(J(MAIN, TEST_SEEDS), 'main')
    if 'abl' in which:
        run(J(ABL, TEST_SEEDS), 'abl')
    if 'alpha' in which:
        jobs = []
        for a in [0.01, 0.02, 0.05, 0.1, 0.2]:
            for m in ['FRESCO', 'Static split CP', 'Gaussian bound']:
                ss = TEST_SEEDS if m == 'FRESCO' else S10
                jobs += [(m, dict(days=TEST_DAYS, alpha=a), s, a, False) for s in ss]
        run(jobs, 'alpha')
    if 'deadline' in which:
        jobs = []
        for D in [15.0, 20.0, 45.0, 60.0]:
            jobs += J(KEY, S10, deadline=D)
        run(jobs, 'deadline')
    if 'load' in which:
        jobs = []
        for g in [10.0, 40.0]:
            jobs += J(KEY, S10, gen_period=g)
        run(jobs, 'load')
    if 'harvest' in which:
        jobs = []
        for h in [0.5, 2.0]:
            jobs += J(KEY, S10, harvest_scale=h)
        run(jobs, 'harvest')
    if 'scale' in which:
        jobs = []
        for n in [50, 200, 300]:
            jobs += J(KEY, TEST_SEEDS[:6], N=n)
        run(jobs, 'scale')
    if 'site' in which:
        jobs = []
        for site in ['miami', 'sandpoint']:
            jobs += J(MAIN, S15, site=site, days=list(range(365)))
        run(jobs, 'site')
    if 'shift' in which:
        run(J(['FRESCO', 'Static split CP', 'Gaussian bound', 'w/o shield', 'FB-MPBatch', 'Lyapunov-DPP'], S15,
              extra_loss_db=6.0, shift_step=1800), 'shift')
    if 'fail' in which:
        run(J(MAIN, S15, fail_frac=0.1), 'fail')
    if 'trace' in which:
        jobs = [(m, dict(days=TEST_DAYS, extra_loss_db=6.0, shift_step=1800), 1003, 0.05, True)
                for m in ['FRESCO', 'Static split CP', 'Gaussian bound']]
        jobs += [(m, dict(days=TEST_DAYS), 1003, 0.05, True) for m in ['FRESCO', 'FB-MPBatch']]
        run(jobs, 'trace')


### `analyze.py`

In [ ]:
%%writefile analyze.py
"""Statistics for the FRESCO evaluation: bootstrap CIs, Wilcoxon-Holm, rank-biserial, Friedman/Nemenyi."""
import json, os, sys
import numpy as np
from scipy import stats

OUT = os.environ.get('FRESCO_OUT', '.')
METRICS = ['aoi', 'paoi95', 'viol', 'miss_released', 'miss_feasible', 'feasible_frac', 'miss_feas_share', 'pdr', 'plr', 'e_bit', 'brownout', 'mean_batch',
           'infeasible', 'shield_raise', 'miscover', 'lost_mc']
LOWER_BETTER = {'aoi': True, 'paoi95': True, 'viol': True, 'miss_released': True, 'pdr': False,
                'e_bit': True, 'brownout': True}


def load(tag):
    p = f'{OUT}/eval_{tag}.json'
    return json.load(open(p)) if os.path.exists(p) else []


def boot_ci(x, n=10000, seed=0):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    bs = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return float(x.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def by_method(rows, key='method'):
    d = {}
    for r in rows:
        d.setdefault(r[key], []).append(r)
    for k in d:
        d[k].sort(key=lambda r: r['seed'])
    return d


def summarize(rows):
    out = {}
    for m, rs in by_method(rows).items():
        out[m] = {k: boot_ci([r.get(k, np.nan) for r in rs]) for k in METRICS}
        out[m]['n'] = len(rs)
    return out


def holm(p):
    p = np.asarray(p)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    m = len(p)
    for k, i in enumerate(order):
        run = max(run, (m - k) * p[i])
        adj[i] = min(1.0, run)
    return adj


def rank_biserial(diff):
    d = np.asarray(diff, float)
    d = d[d != 0]
    if len(d) == 0:
        return 0.0
    rk = stats.rankdata(np.abs(d))
    wp, wm = rk[d > 0].sum(), rk[d < 0].sum()
    return float((wp - wm) / (wp + wm))


def paired_tests(rows, ref, metrics=('aoi', 'viol', 'e_bit', 'paoi95', 'brownout')):
    d = by_method(rows)
    others = [m for m in d if m != ref]
    res = {}
    for met in metrics:
        a = np.array([r[met] for r in d[ref]])
        ps, rbs, deltas = [], [], []
        for m in others:
            b = np.array([r[met] for r in d[m]])
            diff = a - b
            if np.allclose(diff, 0):
                p = 1.0
            else:
                p = float(stats.wilcoxon(a, b, zero_method='wilcox', alternative='two-sided').pvalue)
            ps.append(p)
            rbs.append(rank_biserial(diff))
            deltas.append(float(np.mean(diff)))
        adj = holm(ps)
        res[met] = {m: dict(p=ps[i], p_holm=float(adj[i]), r_rb=rbs[i], delta=deltas[i]) for i, m in enumerate(others)}
    return res


def friedman(rows, metric='aoi'):
    d = by_method(rows)
    ms = list(d)
    X = np.array([[r[metric] for r in d[m]] for m in ms]).T  # episodes x methods
    stat, p = stats.friedmanchisquare(*X.T)
    ranks = np.array([stats.rankdata(row) for row in X])
    mr = ranks.mean(0)
    k, n = len(ms), X.shape[0]
    q = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164}[k]
    cd = q * np.sqrt(k * (k + 1) / (6.0 * n))
    return dict(stat=float(stat), p=float(p), mean_rank={m: float(r) for m, r in zip(ms, mr)}, cd=float(cd), n=n, k=k)


def sweep(rows, param):
    out = {}
    for r in rows:
        key = (r['method'], r.get(f'cfg_{param}'))
        out.setdefault(key, []).append(r)
    res = {}
    for (m, v), rs in out.items():
        res.setdefault(m, {})[str(v)] = {k: boot_ci([x[k] for x in rs]) for k in ['aoi', 'viol', 'miss_released', 'miss_feasible', 'feasible_frac', 'miss_feas_share', 'e_bit', 'brownout', 'pdr', 'miscover', 'paoi95']}
    return res


if __name__ == '__main__':
    R = {}
    main = load('main')
    if main:
        R['main'] = summarize(main)
        R['tests'] = paired_tests(main, 'FRESCO')
        R['friedman_aoi'] = friedman(main, 'aoi')
        R['friedman_viol'] = friedman(main, 'viol')
    abl = load('abl')
    if abl and main:
        full = [r for r in main if r['method'] == 'FRESCO']
        R['abl'] = summarize(abl + full)
        R['abl_tests'] = paired_tests(abl + full, 'FRESCO', metrics=('aoi', 'viol', 'e_bit', 'miss_released', 'brownout'))
    for tag, par in [('deadline', 'deadline'), ('load', 'gen_period'), ('harvest', 'harvest_scale'), ('scale', 'N'), ('alpha', 'alpha')]:
        rows = load(tag)
        if rows:
            R['sweep_' + tag] = sweep(rows, par)
    for tag in ['site', 'shift', 'fail']:
        rows = load(tag)
        if rows:
            if tag == 'site':
                for site in ['miami', 'sandpoint']:
                    rr = [r for r in rows if r.get('cfg_site') == site]
                    R[f'site_{site}'] = summarize(rr)
                    R[f'site_{site}_tests'] = paired_tests(rr, 'FRESCO', metrics=('aoi', 'viol', 'e_bit'))
            else:
                R[tag] = summarize(rows)
                if tag == 'fail':
                    R['fail_tests'] = paired_tests(rows, 'FRESCO', metrics=('aoi', 'viol', 'e_bit'))
    json.dump(R, open(f'{OUT}/analysis.json', 'w'), indent=1)
    if 'main' in R:
        for m, v in R['main'].items():
            print(f"{m:16s} " + ' '.join(f"{k}={v[k][0]:.3f}" for k in ['aoi', 'paoi95', 'viol', 'miss_released', 'pdr', 'e_bit', 'brownout', 'mean_batch']))
        print(R['friedman_aoi'])


### `timing.py`

In [ ]:
%%writefile timing.py
"""Decision-latency and footprint measurements for FRESCO components (single CPU thread)."""
import json, os, time
import numpy as np
import torch
import fresco_sim as fs
import fresco_agents as fa

torch.set_num_threads(1)
OUT = os.environ.get('FRESCO_OUT', '.')


def macs_actor(gat=True, K=10, nb=4):
    m = 0
    if gat:
        m += fa.N_FEAT * 32 + nb * (2 * 32) + nb * 32   # projection, attention logits, weighted sum
    fin = fa.N_FEAT + (32 if gat else 0)
    m += fin * 64 + 64 * 64 + 64 * (fa.N_TAU + fa.N_R)
    return m


def main():
    res = {}
    actor = fa.Actor(True)
    actor.load_state_dict(torch.load(f'{OUT}/actor_fresco.pt'))
    actor.eval()
    K = 10
    X = torch.randn(1, K, fa.N_FEAT)
    A = torch.ones(1, K, K)
    with torch.no_grad():
        for _ in range(50):
            actor(X, A)
        t0 = time.perf_counter()
        n = 2000
        for _ in range(n):
            actor(X, A)
        res['actor_ms_per_call_K10'] = 1e3 * (time.perf_counter() - t0) / n
    res['actor_params'] = int(sum(p.numel() for p in actor.parameters()))
    res['actor_params_nogat'] = int(sum(p.numel() for p in fa.Actor(False).parameters()))
    res['critic_params'] = int(sum(p.numel() for p in fa.Critic().parameters()))
    res['macs_per_head'] = macs_actor(True)
    res['elector_params'] = fa.NeuroFuzzyElector.dim()
    # elector scoring of 100 nodes
    el = fa.NeuroFuzzyElector(np.load(f'{OUT}/elector.npy'))
    Xn = np.random.rand(100, 4)
    t0 = time.perf_counter()
    for _ in range(200):
        el.score(Xn)
    res['elector_ms_100nodes'] = 1e3 * (time.perf_counter() - t0) / 200
    # shield: quantile computation over 4 strata of W=1500
    sh = fa.ConformalShield(0.05, 'aci')
    for k in range(4):
        sh.scores[k].extend(np.random.lognormal(0, 0.5, 1500))
    t0 = time.perf_counter()
    for _ in range(500):
        sh._qc = None
        sh.q()
    res['shield_ms_per_epoch'] = 1e3 * (time.perf_counter() - t0) / 500
    res['shield_memory_kB'] = 4 * 1500 * 8 / 1024
    # whole-episode wall time of the simulator with FRESCO
    cfg = fs.default_config(days=list(range(1, 365, 2)))
    ctrl = fa.MAPPOController(el, actor, 'aci', 0.05)
    t0 = time.perf_counter()
    fa.run_episode(cfg, 1000, ctrl)
    res['episode_s'] = time.perf_counter() - t0
    # GF(256) RLNC decode cost (Gaussian elimination, B=16, payload 24 B) with numpy tables
    exp = np.zeros(512, np.int64)
    log = np.zeros(256, np.int64)
    x = 1
    for i in range(255):
        exp[i] = x
        log[x] = i
        x <<= 1
        if x & 0x100:
            x ^= 0x11D
    exp[255:510] = exp[:255]

    def mul(a, b):
        out = exp[(log[a] + log[b]) % 255]
        return np.where((a == 0) | (b == 0), 0, out)

    def inv(a):
        return exp[(255 - log[a]) % 255]

    B, p = 16, 24
    rng = np.random.default_rng(0)
    t0 = time.perf_counter()
    reps = 50
    for _ in range(reps):
        Mx = rng.integers(0, 256, (B, B + p))
        Mx[:, :B] = rng.integers(1, 256, (B, B))
        for c in range(B):
            piv = next((rr for rr in range(c, B) if Mx[rr, c] != 0), None)
            if piv is None:
                continue
            Mx[[c, piv]] = Mx[[piv, c]]
            Mx[c] = mul(Mx[c], np.full(B + p, inv(Mx[c, c])))
            for rr in range(B):
                if rr != c and Mx[rr, c] != 0:
                    Mx[rr] = Mx[rr] ^ mul(Mx[c], np.full(B + p, Mx[rr, c]))
    res['rlnc_decode_ms_B16'] = 1e3 * (time.perf_counter() - t0) / reps
    res['ctrl_frames_per_round_N100'] = 2 * 100
    json.dump(res, open(f'{OUT}/timing.json', 'w'), indent=1)
    print(res)


if __name__ == '__main__':
    main()


### `fig_diagrams.py`

In [ ]:
%%writefile fig_diagrams.py
"""Grayscale schematic figures: system architecture (Fig. 1) and training/inference workflow (Fig. 2)."""
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch

plt.rcParams.update({'font.family': 'serif', 'font.serif': ['DejaVu Serif'], 'font.size': 7.5,
                     'mathtext.fontset': 'dejavuserif'})


def box(ax, x, y, w, h, text, fc='white', lw=0.9, ls='-', fs=7.2, bold=False):
    p = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.012,rounding_size=0.025', fc=fc, ec='black',
                       lw=lw, ls=ls, zorder=2)
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fs, zorder=3,
            fontweight='bold' if bold else 'normal', linespacing=1.25)
    return (x, y, w, h)


def arrow(ax, p0, p1, ls='-', lw=0.9, rad=0.0, text=None, toff=(0, 0.02), fs=6.5, head=7):
    a = FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=head, lw=lw, ls=ls, color='black',
                        connectionstyle=f'arc3,rad={rad}', zorder=4, shrinkA=0, shrinkB=0)
    ax.add_patch(a)
    if text:
        mx, my = (p0[0] + p1[0]) / 2 + toff[0], (p0[1] + p1[1]) / 2 + toff[1]
        ax.text(mx, my, text, ha='center', va='bottom', fontsize=fs, zorder=5,
                bbox=dict(fc='white', ec='none', pad=0.6))


def R(b):  # right-mid
    return (b[0] + b[2], b[1] + b[3] / 2)


def L(b):
    return (b[0], b[1] + b[3] / 2)


def T(b, f=0.5):
    return (b[0] + b[2] * f, b[1] + b[3])


def B(b, f=0.5):
    return (b[0] + b[2] * f, b[1])


# ------------------------------------------------------------------ Fig. 1
def fig_architecture(path):
    fig = plt.figure(figsize=(7.16, 3.75))
    # (a) network view
    ax = fig.add_axes([0.0, 0.0, 0.33, 1.0])
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.2, 1.0)
    ax.axis('off')
    rng = np.random.default_rng(4)
    pts = rng.uniform(0.05, 0.95, size=(70, 2)) * [1, 0.82] + [0, 0.12]
    heads = np.array([[0.2, 0.78], [0.55, 0.85], [0.82, 0.62], [0.3, 0.45], [0.7, 0.35]])
    for p in pts:
        d = np.linalg.norm(heads - p, axis=1)
        h = heads[np.argmin(d)]
        ax.plot([p[0], h[0]], [p[1], h[1]], color='0.75', lw=0.4, zorder=1)
    ax.scatter(pts[:, 0], pts[:, 1], s=6, c='0.55', zorder=2, lw=0)
    ax.scatter(heads[:, 0], heads[:, 1], s=38, marker='s', c='black', zorder=3)
    sink = np.array([0.5, -0.04])
    ax.scatter([sink[0]], [sink[1]], s=90, marker='^', c='white', edgecolors='black', lw=1.0, zorder=4)
    ax.text(sink[0] + 0.07, sink[1] - 0.01, 'sink', fontsize=7, va='center')
    # three node-disjoint paths from head 1
    h = heads[1]
    paths = [[h, [0.48, 0.62], [0.46, 0.32], sink], [h, [0.66, 0.6], [0.62, 0.2], sink],
             [h, [0.38, 0.7], [0.18, 0.3], [0.36, 0.08], sink]]
    styles = ['-', '--', ':']
    for p, st in zip(paths, styles):
        p = np.array(p)
        ax.plot(p[:, 0], p[:, 1], ls=st, color='black', lw=1.1, zorder=3)
        ax.scatter(p[1:-1, 0], p[1:-1, 1], s=14, c='white', edgecolors='black', lw=0.7, zorder=4)
    ax.annotate('', xy=sink + [0.0, 0.035], xytext=sink + [0.0, 0.034])
    ax.text(0.55, 0.93, r'head $h$', fontsize=7, ha='left')
    # legend drawn as separate rows below the field
    ly = -0.105
    ax.scatter([0.04], [ly], s=6, c='0.55', lw=0)
    ax.text(0.07, ly, 'member', va='center', fontsize=6.5)
    ax.scatter([0.30], [ly], s=24, marker='s', c='black')
    ax.text(0.33, ly, 'head', va='center', fontsize=6.5)
    ax.scatter([0.50], [ly], s=14, c='white', edgecolors='black', lw=0.7)
    ax.text(0.53, ly, 'relay', va='center', fontsize=6.5)
    ax.text(0.72, ly, '', va='center')
    ly2 = -0.175
    for k, (st, lab) in enumerate(zip(['-', '--', ':'], [r'$\pi_1$', r'$\pi_2$', r'$\pi_3$'])):
        x0 = 0.04 + 0.25 * k
        ax.plot([x0, x0 + 0.1], [ly2, ly2], color='black', ls=st, lw=1.1)
        ax.text(x0 + 0.12, ly2, lab, va='center', fontsize=6.5)
    ax.text(0.02, 0.985, '(a)', fontsize=8, fontweight='bold', va='top')

    # (b) pipeline
    ax = fig.add_axes([0.35, 0.0, 0.65, 1.0])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(0.0, 0.985, '(b)', fontsize=8, fontweight='bold', va='top')
    g = '0.9'
    # round level (top row)
    b1 = box(ax, 0.04, 0.82, 0.25, 0.11, 'Node state $\\mathbf{x}_i$\n(storage, harvest\nforecast, ETX, load)')
    b2 = box(ax, 0.375, 0.82, 0.25, 0.11, 'Neuro-fuzzy elector\nTSK, 81 rules\n(every round)', fc=g, bold=False)
    b3 = box(ax, 0.71, 0.82, 0.25, 0.11, 'Heads $\\mathcal{H}$, clusters $\\mathcal{M}_h$,\n$k$ disjoint min-ETX\npaths per head')
    arrow(ax, R(b1), L(b2))
    arrow(ax, R(b2), L(b3))
    ax.text(0.5, 0.955, 'Round level (every 300 s)', ha='center', fontsize=7, style='italic')
    # batch level (middle)
    b4 = box(ax, 0.04, 0.50, 0.21, 0.13, 'Head buffer\n$B$ updates,\noldest age $a$')
    b5 = box(ax, 0.30, 0.50, 0.19, 0.13, 'GAT encoder\nover head graph', fc=g)
    b6 = box(ax, 0.54, 0.50, 0.19, 0.13, 'Policy $\\pi_\\theta$\nrelease age $\\tau$\nrepair level $r$', fc=g)
    b7 = box(ax, 0.78, 0.50, 0.19, 0.13, 'Conformal shield\n$a+q_{r,t}\\,\\hat d(r)\\leq D$', fc=g)
    arrow(ax, R(b4), L(b5))
    arrow(ax, R(b5), L(b6))
    arrow(ax, R(b6), L(b7))
    arrow(ax, B(b3, 0.66), T(b7, 0.5), text=None)
    ax.plot([0.75, 0.75], [0.82, 0.70], 'k-', lw=0.9, zorder=4)
    ax.plot([0.75, 0.145], [0.70, 0.70], 'k-', lw=0.9, zorder=4)
    arrow(ax, (0.145, 0.70), T(b4, 0.5))
    ax.text(0.45, 0.715, 'Batch level (event driven)', ha='center', fontsize=7, style='italic')
    # transmission level (bottom)
    b8 = box(ax, 0.78, 0.18, 0.19, 0.15, 'Systematic RLNC\n$B$ originals + $r$\nrepairs; split $n_j$')
    b9 = box(ax, 0.48, 0.18, 0.22, 0.15, 'Duty-cycled\n802.15.4 relays\n$\\pi_1,\\ldots,\\pi_k$')
    b10 = box(ax, 0.18, 0.18, 0.22, 0.15, 'Sink decoder\ndelivery times $T_i$\nAoI $\\Delta_n(t)$')
    arrow(ax, B(b7), T(b8), text='release')
    arrow(ax, L(b8), R(b9))
    arrow(ax, L(b9), R(b10))
    # feedback: sink -> shield calibration (dashed)
    ax.plot([0.29, 0.29], [0.18, 0.07], 'k--', lw=0.9, zorder=4)
    ax.plot([0.29, 0.995], [0.07, 0.07], 'k--', lw=0.9, zorder=4)
    ax.plot([0.995, 0.995], [0.07, 0.565], 'k--', lw=0.9, zorder=4)
    arrow(ax, (0.995, 0.565), (0.97, 0.565), ls='--')
    ax.text(0.64, 0.085, 'feedback: ratio $s_i=T_i/\\hat d(r)$, ACI update of $\\alpha_{t}$', ha='center',
            fontsize=6.5, bbox=dict(fc='white', ec='none', pad=0.6))
    # AoI feedback to policy reward (dash-dot)
    ax.plot([0.24, 0.24], [0.33, 0.43], 'k-.', lw=0.8, zorder=4)
    ax.plot([0.24, 0.515], [0.43, 0.43], 'k-.', lw=0.8, zorder=4)
    arrow(ax, (0.515, 0.43), (0.635, 0.50), ls='-.', lw=0.8)
    ax.text(0.375, 0.445, 'reward $-\\Delta/D$, costs $c^E, c^R$', ha='center', fontsize=6.5,
            bbox=dict(fc='white', ec='none', pad=0.4))
    fig.savefig(path, dpi=300)
    plt.close(fig)


# ------------------------------------------------------------------ Fig. 2
def fig_training(path):
    fig, ax = plt.subplots(figsize=(7.16, 3.2))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    g = '0.9'
    ax.text(0.125, 0.965, 'Stage 1: elector search (CEM)', ha='center', fontsize=7.5, fontweight='bold')
    s1 = box(ax, 0.03, 0.70, 0.19, 0.16, 'Sample $\\boldsymbol{\\phi}^{(m)}\\sim$\n$\\mathcal{N}(\\boldsymbol{\\mu},\\mathrm{diag}\\,\\boldsymbol{\\sigma}^2)$')
    s2 = box(ax, 0.03, 0.42, 0.19, 0.16, 'Episodes with fixed\nrelease rule and\nACI shield')
    s3 = box(ax, 0.03, 0.12, 0.19, 0.16, 'Elite refit of\n$(\\boldsymbol{\\mu},\\boldsymbol{\\sigma})$')
    arrow(ax, B(s1), T(s2))
    arrow(ax, B(s2), T(s3), text='fitness $F$', toff=(0.045, -0.01))
    ax.plot([0.03, 0.012, 0.012], [0.20, 0.20, 0.78], 'k-', lw=0.9, zorder=4)
    arrow(ax, (0.012, 0.78), (0.03, 0.78))
    ax.text(0.66, 0.965, 'Stage 2: Lagrangian GAT-MAPPO with the shield in the loop', ha='center', fontsize=7.5,
            fontweight='bold')
    e0 = box(ax, 0.27, 0.70, 0.14, 0.16, 'Frozen elector\n$\\boldsymbol{\\phi}^{\\star}$', fc=g)
    e7 = box(ax, 0.27, 0.42, 0.14, 0.16, 'Actor $\\pi_\\theta$\n(GAT + MLP)', fc=g)
    e1 = box(ax, 0.48, 0.70, 0.19, 0.16, 'Simulator rollout\n(1 compressed day)')
    e2 = box(ax, 0.75, 0.70, 0.22, 0.16, 'Semi-Markov buffer\n$(o,\\tau,r,R,C^E,C^R,k)$')
    e3 = box(ax, 0.75, 0.42, 0.22, 0.16, 'Critic heads\n$V_r,\\,V_E,\\,V_R$ and GAE')
    e4 = box(ax, 0.48, 0.42, 0.19, 0.16, 'Lagrangian advantage\n$\\hat A_r-\\lambda_E\\hat A_E-\\lambda_R\\hat A_R$')
    e5 = box(ax, 0.495, 0.12, 0.16, 0.16, 'Clipped PPO loss\n$\\mathcal{L}(\\theta)$', fc=g)
    e6 = box(ax, 0.7375, 0.12, 0.245, 0.16, 'Dual ascent\n$\\lambda\\leftarrow[\\lambda+\\eta_\\lambda\\,\\mathrm{clip}(\\hat J/\\epsilon-1)]_+$', fs=6.7)
    arrow(ax, R(s1), L(e0))
    ax.text(0.245, 0.80, '$\\boldsymbol{\\mu}$', ha='center', fontsize=6.5)
    arrow(ax, R(e0), L(e1))
    arrow(ax, (0.41, 0.55), (0.50, 0.70), text=None)
    ax.text(0.475, 0.605, '$\\tau, r$', fontsize=6.8, ha='center')
    arrow(ax, R(e1), L(e2))
    arrow(ax, B(e2), T(e3))
    arrow(ax, L(e3), R(e4))
    arrow(ax, B(e4), T(e5))
    arrow(ax, B(e3), T(e6), text='$\\hat J_E,\\hat J_R$', toff=(0.045, -0.015))
    arrow(ax, L(e6), R(e5))
    ax.text(0.696, 0.21, '$\\lambda_E,\\lambda_R$', ha='center', va='bottom', fontsize=6.3)
    # dashed reverse-mode gradient from the loss back to the actor parameters
    ax.plot([0.495, 0.34], [0.20, 0.20], 'k--', lw=1.0, zorder=4)
    arrow(ax, (0.34, 0.20), (0.34, 0.42), ls='--', lw=1.0)
    ax.text(0.33, 0.31, '$\\nabla_\\theta\\mathcal{L}$\n(backprop.)', ha='right', va='center', fontsize=6.5)
    ax.text(0.62, 0.035, 'Inference: deterministic arg max of both policy heads; shield and calibrator stay online',
            ha='center', fontsize=6.8, style='italic')
    fig.savefig(path, dpi=300)
    plt.close(fig)


if __name__ == '__main__':
    fig_architecture(os.path.join(os.environ.get('FRESCO_FIG', '.'), 'fig1_architecture.png'))
    fig_training(os.path.join(os.environ.get('FRESCO_FIG', '.'), 'fig2_training.png'))


### `make_figures.py`

In [ ]:
%%writefile make_figures.py
"""Grayscale result figures for the manuscript (300 dpi, column or page width)."""
import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import fresco_agents as fa

OUT = os.environ.get('FRESCO_OUT', '.')
FIG = os.environ.get('FRESCO_FIG', '.')
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['DejaVu Serif'], 'font.size': 7.5,
                     'axes.linewidth': 0.6, 'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
                     'axes.labelsize': 7.5, 'legend.fontsize': 6.5, 'mathtext.fontset': 'dejavuserif',
                     'lines.linewidth': 1.0, 'legend.frameon': False})
A = json.load(open(f'{OUT}/analysis.json')) if os.path.exists(f'{OUT}/analysis.json') else {}
MARK = {'FRESCO': ('*', 'black', 9), 'LEACH-Direct': ('v', '0.55', 5), 'HEED-MMSPEED': ('^', '0.55', 5),
        'FB-MPBatch': ('s', '0.35', 5), 'SCM-RLNC': ('D', '0.35', 4.5), 'EA-AgeThr': ('o', '0.2', 5),
        'Lyapunov-DPP': ('P', '0.45', 5.5), 'IQL-DQN': ('X', '0.6', 5.5), 'MAPPO-Penalty': ('h', '0.6', 5.5)}
LS = {'FRESCO': '-', 'FB-MPBatch': '--', 'SCM-RLNC': '-.', 'EA-AgeThr': ':', 'Lyapunov-DPP': (0, (5, 1, 1, 1, 1, 1))}


def save(fig, name):
    fig.savefig(f'{FIG}/{name}', dpi=300)
    plt.close(fig)


def fig_main():
    S = A['main']
    fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.75), gridspec_kw=dict(width_ratios=[1.15, 1]))
    ax = axs[0]
    for m, (mk, c, ms) in MARK.items():
        if m not in S:
            continue
        x, xl, xh = S[m]['e_bit']
        y, yl, yh = S[m]['aoi']
        ax.errorbar(x, y, xerr=[[x - xl], [xh - x]], yerr=[[y - yl], [yh - y]], fmt=mk, color=c, mec='black',
                    mew=0.5, ms=ms, elinewidth=0.6, capsize=1.5, label=m, zorder=3 if m == 'FRESCO' else 2)
    ax.set_xlabel('Energy per delivered bit ($\\mu$J/bit)')
    ax.set_ylabel('Average age of information (s)')
    ax.grid(True, lw=0.3, color='0.85')
    ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), handletextpad=0.3, borderaxespad=0.2)
    ax.text(0.02, 0.97, '(a)', transform=ax.transAxes, va='top', fontweight='bold')
    ax = axs[1]
    order = [m for m in MARK if m in S]
    xs = np.arange(len(order))
    v = [S[m]['viol'][0] * 100 for m in order]
    lo = [100 * (S[m]['viol'][0] - S[m]['viol'][1]) for m in order]
    hi = [100 * (S[m]['viol'][2] - S[m]['viol'][0]) for m in order]
    ms_ = [S[m]['miss_released'][0] * 100 for m in order]
    ax.bar(xs - 0.2, v, 0.38, yerr=[lo, hi], color=['black' if m == 'FRESCO' else '0.65' for m in order],
           ec='black', lw=0.4, error_kw=dict(lw=0.6, capsize=1.5), label='all generated updates')
    ax.bar(xs + 0.2, ms_, 0.38, color='white', ec='black', lw=0.5, hatch='////', label='released updates')
    ax.axhline(5, color='black', ls='--', lw=0.7, label='target $\\alpha=5\\%$')
    ax.set_xticks(xs)
    ax.set_xticklabels(order, rotation=40, ha='right', fontsize=6.3)
    ax.set_ylabel('Deadline-miss rate (%)')
    ax.legend(loc='upper left', bbox_to_anchor=(0.0, 1.0))
    ax.set_ylim(0, max(v) * 1.35)
    ax.text(0.98, 0.97, '(b)', transform=ax.transAxes, va='top', ha='right', fontweight='bold')
    fig.tight_layout(w_pad=0.8)
    save(fig, 'fig3_main.png')


def fig_cd():
    F = A['friedman_aoi']
    mr = F['mean_rank']
    cd = F['cd']
    items = sorted(mr.items(), key=lambda kv: kv[1])
    k = len(items)
    fig, ax = plt.subplots(figsize=(3.5, 1.75))
    lo, hi = 1, k
    ax.set_xlim(lo - 0.3, hi + 0.3)
    ax.set_ylim(0.12, 1.0)
    ax.axis('off')
    y0 = 0.80
    ax.plot([lo, hi], [y0, y0], 'k-', lw=0.8)
    for t in range(lo, hi + 1):
        ax.plot([t, t], [y0, y0 + 0.04], 'k-', lw=0.7)
        ax.text(t, y0 + 0.07, str(t), ha='center', fontsize=6.5)
    ax.plot([lo, lo + cd], [0.95, 0.95], 'k-', lw=1.2)
    ax.text(lo + cd + 0.15, 0.95, f'CD = {cd:.2f}', ha='left', va='center', fontsize=6.3)
    half = (k + 1) // 2
    for i, (m, r) in enumerate(items):
        if i < half:
            yy = y0 - 0.12 - 0.11 * i
            ax.plot([r, r, lo - 0.05], [y0, yy, yy], 'k-', lw=0.6)
            ax.text(lo - 0.1, yy, f'{m} ({r:.2f})', ha='right', va='center', fontsize=6.2)
        else:
            j = i - half
            yy = y0 - 0.12 - 0.11 * (k - 1 - i)
            ax.plot([r, r, hi + 0.05], [y0, yy, yy], 'k-', lw=0.6)
            ax.text(hi + 0.1, yy, f'({r:.2f}) {m}', ha='left', va='center', fontsize=6.2)
    # cliques: groups whose rank span < cd
    ranks = [r for _, r in items]
    cl = []
    for i in range(k):
        j = i
        while j + 1 < k and ranks[j + 1] - ranks[i] < cd:
            j += 1
        if j > i and not any(a <= i and j <= b for a, b in cl):
            cl.append((i, j))
    for n_, (i, j) in enumerate(cl):
        yy = y0 - 0.06 - 0.035 * n_
        ax.plot([ranks[i] - 0.03, ranks[j] + 0.03], [yy, yy], 'k-', lw=2.0)
    fig.subplots_adjust(left=0.30, right=0.70, top=0.98, bottom=0.02)
    save(fig, 'fig4_cd.png')


def fig_calibration():
    sw = A['sweep_alpha']
    fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.45), gridspec_kw=dict(width_ratios=[1, 1, 1.15]))
    ax = axs[0]
    marks = {'FRESCO': ('o', '-', 'ACI (FRESCO)'), 'Static split CP': ('s', '--', 'static split CP'),
             'Gaussian bound': ('^', ':', 'Gaussian bound')}
    for m, (mk, ls, lab) in marks.items():
        if m not in sw:
            continue
        al = sorted(sw[m], key=float)
        x = [float(a) for a in al]
        y = [sw[m][a]['miscover'][0] for a in al]
        yl = [sw[m][a]['miscover'][0] - sw[m][a]['miscover'][1] for a in al]
        yh = [sw[m][a]['miscover'][2] - sw[m][a]['miscover'][0] for a in al]
        ax.errorbar(x, y, yerr=[yl, yh], marker=mk, ls=ls, color='black', mfc='white' if m != 'FRESCO' else 'black',
                    ms=3.5, capsize=1.5, elinewidth=0.6, label=lab)
    xx = np.array([0.008, 0.25])
    ax.plot(xx, xx, color='0.6', lw=0.8, label='$y=\\alpha$')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Target level $\\alpha$')
    ax.set_ylabel('Observed miscoverage')
    ax.grid(True, which='both', lw=0.3, color='0.88')
    ax.legend(loc='lower right', fontsize=6.0)
    ax.set_title('(a)', loc='left', fontsize=7.5, fontweight='bold')
    # (b) age versus overall miss rate: FRESCO across alpha, baselines at their tuned points
    ax = axs[1]
    f = sw['FRESCO']
    al = sorted(f, key=float)
    ax.plot([100 * f[a]['viol'][0] for a in al], [f[a]['aoi'][0] for a in al], 'k-o', ms=3.5,
            label='FRESCO, $\\alpha$ sweep', zorder=3)
    for a in al:
        off = {0.01: ((-5, 0), 'right', 'center'), 0.02: ((-5, 0), 'right', 'center'),
               0.05: ((-5, 0), 'right', 'center'), 0.1: ((0, -6), 'center', 'top'), 0.2: ((5, 0), 'left', 'center')}
        xy, ha, va = off[round(float(a), 2)]
        ax.annotate(f'$\\alpha$={float(a):g}', (100 * f[a]['viol'][0], f[a]['aoi'][0]), textcoords='offset points',
                    xytext=xy, ha=ha, va=va, fontsize=5.6)
    S = A['main']
    for m, (mk, c, ms) in MARK.items():
        if m in ('FRESCO', 'LEACH-Direct', 'HEED-MMSPEED', 'FB-MPBatch') or m not in S:
            continue
        ax.plot(100 * S[m]['viol'][0], S[m]['aoi'][0], mk, color=c, mec='black', mew=0.5, ms=ms * 0.8, ls='none',
                label=m)
    ax.set_xlabel('Deadline-miss rate, all updates (%)')
    ax.set_ylabel('Average age (s)')
    ax.grid(True, lw=0.3, color='0.88')
    ax.set_xlim(4.0, 12.5)
    ax.set_ylim(22.5, 29.8)
    ax.legend(loc='upper right', fontsize=5.6, handletextpad=0.2, labelspacing=0.25)
    ax.set_title('(b)', loc='left', fontsize=7.5, fontweight='bold')
    ax = axs[2]
    tr = json.load(open(f'{OUT}/eval_trace.json'))
    styles = {'FRESCO': ('-', 'black', 'ACI (FRESCO)'), 'Static split CP': ('--', '0.35', 'static split CP'),
              'Gaussian bound': (':', 'black', 'Gaussian bound')}
    for r in tr:
        if r.get('cfg_shift_step') is None or r['method'] not in styles:
            continue
        ls, c, lab = styles[r['method']]
        t = np.array(r['trace']['t']) / 3600 * 24
        v = np.array(r['trace']['q'], float) * 100
        ax.plot(t, v, ls=ls, color=c, lw=1.0, label=lab)
    ax.axvline(12, color='0.4', lw=0.7)
    ax.text(11.6, 0.55, '+6 dB loss on\nevery link from\n12 h onwards', transform=ax.get_xaxis_transform(),
            va='center', ha='right', fontsize=6.0)
    ax.axhline(5, color='0.6', lw=0.8, ls='-')
    ax.set_xlabel('Time of compressed day (h)')
    ax.set_ylabel('Rolling miscoverage (%)')
    ax.set_xlim(0, 24)
    ax.grid(True, lw=0.3, color='0.88')
    ax.set_ylim(-3, 112)
    ax.legend(loc='upper left', fontsize=5.6, ncol=1, handlelength=1.6)
    ax.set_title('(c)', loc='left', fontsize=7.5, fontweight='bold')
    fig.tight_layout(w_pad=1.0)
    save(fig, 'fig5_calibration.png')


def _sweep_panel(ax, key, param_vals, metric, xlabel, ylabel, scale=1.0, base=None, xlog=False):
    sw = A[key]
    main = A['main']
    for m, ls in LS.items():
        xs, ys, lo, hi = [], [], [], []
        for v in param_vals:
            if base is not None and v == base:
                st = main.get(m, {}).get(metric)
            else:
                st = sw.get(m, {}).get(str(v))
                st = st[metric] if st else None
            if st is None:
                continue
            xs.append(v)
            ys.append(st[0] * scale)
            lo.append((st[0] - st[1]) * scale)
            hi.append((st[2] - st[0]) * scale)
        mk = MARK[m][0]
        ax.errorbar(xs, ys, yerr=[lo, hi], ls=ls, marker=mk, ms=3.5, color='black' if m == 'FRESCO' else '0.4',
                    mfc='black' if m == 'FRESCO' else 'white', mec='black', mew=0.5, elinewidth=0.5, capsize=1.2,
                    label=m)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if xlog:
        ax.set_xscale('log', base=2)
        ax.set_xticks(param_vals)
        ax.set_xticklabels([str(p) for p in param_vals])
    ax.grid(True, lw=0.3, color='0.88')


def fig_sensitivity():
    fig, axs = plt.subplots(2, 2, figsize=(7.16, 4.6))
    _sweep_panel(axs[0, 0], 'sweep_deadline', [15.0, 20.0, 30.0, 45.0, 60.0], 'viol', 'Deadline $D$ (s)',
                 'Deadline-miss rate (%)', 100, base=30.0)
    _sweep_panel(axs[0, 1], 'sweep_load', [10.0, 20.0, 40.0], 'aoi', 'Sampling period $T_{\\mathrm{g}}$ (s)',
                 'Average age (s)', 1, base=20.0)
    _sweep_panel(axs[1, 0], 'sweep_harvest', [0.5, 1.0, 2.0], 'aoi', 'Harvest scale (relative panel area)',
                 'Average age (s)', 1, base=1.0, xlog=True)
    _sweep_panel(axs[1, 1], 'sweep_scale', [50, 100, 200, 300], 'aoi', 'Number of nodes $N$',
                 'Average age (s)', 1, base=100)
    for ax, lab in zip(axs.flat, ['(a)', '(b)', '(c)', '(d)']):
        ax.set_title(lab, loc='left', fontsize=7.5, fontweight='bold')
    h, l = axs[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=5, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.06, 1, 1), h_pad=1.0, w_pad=1.2)
    save(fig, 'fig6_sensitivity.png')


def fig_diurnal():
    tr = json.load(open(f'{OUT}/eval_trace.json'))
    r = next(x for x in tr if x['method'] == 'FRESCO' and x.get('cfg_shift_step') is None)
    rb = next(x for x in tr if x['method'] == 'FB-MPBatch' and x.get('cfg_shift_step') is None)
    fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.45))
    ax = axs[0]
    t = np.array(r['trace']['t']) / 3600 * 24
    ax.fill_between(t, 0, np.array(r['trace']['ghi']), color='0.88', lw=0, label='irradiance $G(t)$')
    ax.set_ylabel('Irradiance (W/m$^2$)')
    ax.set_xlabel('Time of compressed day (h)')
    ax2 = ax.twinx()
    ax2.plot(t, 100 * np.array(r['trace']['store_ch']), 'k-', lw=1.0, label='head storage, FRESCO')
    ax2.plot(np.array(rb['trace']['t']) / 3600 * 24, 100 * np.array(rb['trace']['store_ch']), 'k--', lw=0.9,
             label='head storage, FB-MPBatch')
    ax2.set_ylabel('Mean head storage (% of $C$)')
    ax2.set_ylim(0, 140)
    ax2.set_yticks([0, 20, 40, 60, 80, 100])
    ax.set_xlim(0, 24)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=6.0)
    ax.set_ylim(0, max(np.array(r['trace']['ghi']).max() * 1.6, 10))
    ax.set_title('(a)', loc='left', fontsize=7.5, fontweight='bold')
    ax = axs[1]
    d = np.array(r['trace']['decisions'], float)
    hrs = d[:, 0] / 3600 * 24
    bins = np.arange(0, 25, 2)
    idx = np.digitize(hrs, bins) - 1
    tau = np.array(fa.TAUS)[d[:, 1].astype(int)]
    rr = d[:, 2]
    mt = [tau[idx == b].mean() if (idx == b).any() else np.nan for b in range(len(bins) - 1)]
    mr = [rr[idx == b].mean() if (idx == b).any() else np.nan for b in range(len(bins) - 1)]
    cx = bins[:-1] + 1
    ax.plot(cx, mt, 'k-o', ms=3.5, label='mean release age $\\tau$ (s)')
    ax.set_ylabel('Release age $\\tau$ (s)')
    ax.set_xlabel('Time of compressed day (h)')
    ax.set_xlim(0, 24)
    ax.set_ylim(0, max(np.nanmax(mt) * 1.4, 5))
    ax2 = ax.twinx()
    ax2.bar(cx, mr, width=1.4, color='0.8', ec='black', lw=0.4, label='mean proposed repair level $r$', zorder=0)
    ax2.set_ylim(0, 3.2)
    ax2.set_ylabel('Proposed repair level $r$')
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=6.0)
    ax.set_title('(b)', loc='left', fontsize=7.5, fontweight='bold')
    fig.tight_layout(w_pad=1.5)
    save(fig, 'fig7_diurnal.png')


def fig_membership():
    el = fa.NeuroFuzzyElector(np.load(f'{OUT}/elector.npy'))
    prior = fa.NeuroFuzzyElector()
    names = ['storage $E_i/C$', 'harvest forecast', 'mean neighbour ETX (scaled)', 'relay load (scaled)']
    fig, axs = plt.subplots(1, 4, figsize=(7.16, 1.95), sharey=True)
    x = np.linspace(0, 1, 200)
    for k, ax in enumerate(axs):
        for m, ls in zip(range(3), ['-', '--', ':']):
            ax.plot(x, np.exp(-(x - el.c[k, m]) ** 2 / (2 * el.s[k, m] ** 2)), ls=ls, color='black', lw=1.0,
                    label=['learned: low', 'learned: medium', 'learned: high'][m])
        for m, ls in zip(range(3), ['-', '--', ':']):
            ax.plot(x, np.exp(-(x - prior.c[k, m]) ** 2 / (2 * prior.s[k, m] ** 2)), ls=ls, color='0.7', lw=0.7,
                    label='prior (grey)' if m == 0 else None)
        ax.set_xlabel(names[k])
        ax.set_xlim(0, 1)
        ax.set_xticks([0, 0.5, 1])
        ax.set_xticklabels(['0', '0.5', '1'])
        ax.set_ylim(0, 1.05)
        ax.grid(True, lw=0.3, color='0.9')
    axs[0].set_ylabel('Membership')
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.0), fontsize=6.3)
    fig.tight_layout(rect=(0, 0, 1, 0.88), w_pad=1.2)
    save(fig, 'fig8_membership.png')


def fig_training():
    L = json.load(open(f'{OUT}/trainlog_fresco.json'))['log']
    L2 = json.load(open(f'{OUT}/trainlog_abl_nolag.json'))['log']
    fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.0))
    def sm(v, w=5):
        v = np.array(v, float)
        return np.convolve(v, np.ones(w) / w, mode='valid')
    ep = np.arange(len(L))
    for lg, ls, lab in [(L, '-', 'Lagrangian (FRESCO)'), (L2, '--', 'fixed penalty')]:
        e = np.arange(len(lg))
        axs[0].plot(e[4:], sm([x['aoi'] for x in lg]), ls=ls, color='black', label=lab)
        axs[1].plot(e[4:], sm([x['cost_e'] for x in lg]), ls=ls, color='black', label=lab)
    axs[1].axhline(json.load(open(f'{OUT}/selected_eps.json'))['eps_e'], color='0.6', lw=0.8, label='budget $\\epsilon_E$')
    axs[2].plot(ep, [x['lmb_e'] for x in L], 'k-', label='$\\lambda_E$')
    axs[2].plot(ep, [x['lmb_r'] for x in L], 'k:', label='$\\lambda_R$')
    axs[0].set_ylabel('Training-episode age (s)')
    axs[1].set_ylabel('Energy cost $\\hat J_E$')
    axs[2].set_ylabel('Multiplier value')
    for ax, lab in zip(axs, ['(a)', '(b)', '(c)']):
        ax.set_xlabel('Training episode')
        ax.grid(True, lw=0.3, color='0.88')
        ax.set_title(lab, loc='left', fontsize=7.5, fontweight='bold')
    axs[0].set_ylim(axs[0].get_ylim()[0], axs[0].get_ylim()[1] + 4)
    axs[0].legend(loc='upper right', fontsize=6.0)
    axs[1].set_ylim(axs[1].get_ylim()[0], 0.86)
    axs[1].legend(loc='upper right', fontsize=6.0)
    axs[2].legend(loc='center right', fontsize=6.5)
    fig.tight_layout(w_pad=1.0)
    save(fig, 'fig9_training.png')


if __name__ == '__main__':
    import sys
    which = sys.argv[1:] or ['main', 'cd', 'cal', 'sens', 'diurnal', 'mf', 'train']
    for w, f in [('main', fig_main), ('cd', fig_cd), ('cal', fig_calibration), ('sens', fig_sensitivity),
                 ('diurnal', fig_diurnal), ('mf', fig_membership), ('train', fig_training)]:
        if w in which:
            try:
                f()
                print('ok', w)
            except Exception as e:
                import traceback
                traceback.print_exc()


## 1. Stage 1: neuro-fuzzy elector (cross-entropy method) and baseline tuning

In [ ]:
!python train_all.py cem
!python train_all.py tune

## 2. Stage 2: Lagrangian GAT-MAPPO for three energy budgets, selection on tuning seeds

In [ ]:
!python train_all.py eps
!python select_eps.py

## 3. Ablation variants and learned baselines

In [ ]:
!python train_all.py rl

## 4. Evaluation on held-out seeds, days and sites

In [ ]:
!python run_eval.py main abl alpha deadline load harvest scale site shift fail trace

## 5. Statistics, timing and figures

In [ ]:
!python analyze.py
!python timing.py
!python fig_diagrams.py
!python make_figures.py

In [ ]:
import json, pandas as pd
A = json.load(open('runs/analysis.json'))
rows = {m: {k: v[k][0] for k in ['aoi','paoi95','viol','miss_released','pdr','e_bit','brownout']} for m, v in A['main'].items()}
pd.DataFrame(rows).T.round(3)

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figs/*.png')):
    print(f); display(Image(f, width=700))